Stern-Gerlach Experiment Simulation

We will simulate the motion of silver atoms through the Stern-Gerlach apparatus. The simulation will include:

    Constants and Parameters: Define the values for physical constants (silver atom mass, Bohr magneton, Boltzmann constant) and experimental parameters (temperature, magnetic field gradient, magnet length, distance to the screen).
    Beam Velocity Calculation: Use the average kinetic energy of atoms in the effusive beam to determine their velocity.
    Trajectory Calculation: Determine how atoms are deflected as they pass through the magnetic field and then travel to the screen.
    Animation: Visualize the atom trajectories to show the beam splitting.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Physical constants and experimental parameters ---

T = 1273.15  # Temperature in Kelvin (1000°C = 1273.15K)
nabla_B = 10  # Magnetic field gradient in T/m
L = 1  # Magnet length in meters
D = 1  # Distance from magnet to screen in meters
mu_B = 9.274e-24  # Bohr magneton in J/T
k_B = 1.381e-23  # Boltzmann constant in J/K
m_Ag = 1.7912e-25 # Mass of a silver atom in kg (107.8682 u * 1.660539e-27 kg/u)

print(f"Temperature (T): {T} K")
print(f"Magnetic field gradient (∇B): {nabla_B} T/m")
print(f"Magnet length (L): {L} m")
print(f"Distance to screen (D): {D} m")
print(f"Bohr magneton (μ_B): {mu_B} J/T")
print(f"Boltzmann constant (k_B): {k_B} J/K")
print(f"Mass of Silver atom (m_Ag): {m_Ag} kg")

# --- Calculate beam velocity (using average kinetic energy for the effusive beam) ---
v = np.sqrt((4 * k_B * T) / m_Ag)
print(f"\nAverage silver beam velocity (v): {v:.2f} m/s")

# --- Calculate acceleration within the magnetic field (for one spin state) ---
# For simplicity, we assume atoms have spin +1/2 or -1/2 in the z-direction.
# The force in z is F_z = +/- mu_B * nabla_B
a_z = (mu_B * nabla_B) / m_Ag # Magnitude of acceleration in the z-axis
print(f"Acceleration in z-axis (magnitude): {a_z:.2e} m/s^2")

# --- Calculate total displacement on the screen (as a reference point) ---
delta_z_total = (mu_B * nabla_B) / (2 * m_Ag * v**2) * L * (L + 2 * D)
print(f"Vertical displacement on the screen for one beam (Δz_t): {delta_z_total:.4e} m ({delta_z_total * 1000:.3f} mm)")

# The user calculated d = 1.978mm, which is 2 * delta_z_total. Let's verify.
separation_distance_calculated = (mu_B * nabla_B) / (4 * k_B * T) * L * (L + 2 * D)
print(f"Total separation of the two beams (using direct formula with T): {separation_distance_calculated:.4e} m ({separation_distance_calculated * 1000:.3f} mm)")

In [ ]:

def get_trajectory(spin_direction, num_points=200):
    t_L = L / v  # Time to traverse the magnet
    t_D = D / v  # Time to travel from magnet to screen

    # Time points for the trajectory
    times_in_magnet = np.linspace(0, t_L, num_points // 2)
    times_after_magnet = np.linspace(t_L, t_L + t_D, num_points // 2)

    # X positions
    x_in_magnet = v * times_in_magnet
    x_after_magnet = v * times_after_magnet

    # Z positions (vertical deflection)
    # Inside the magnet:
    z_in_magnet = 0.5 * (spin_direction * a_z) * times_in_magnet**2

    # Vertical velocity upon exiting the magnet
    v_z_at_exit = (spin_direction * a_z) * t_L
    # Vertical displacement upon exiting the magnet
    z_at_exit = 0.5 * (spin_direction * a_z) * t_L**2

    # After the magnet (uniform rectilinear motion in z)
    z_after_magnet = z_at_exit + v_z_at_exit * (times_after_magnet - t_L)

    return np.concatenate((x_in_magnet, x_after_magnet)), np.concatenate((z_in_magnet, z_after_magnet))


# --- Animation Setup ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(-0.1, L + D + 0.1) # A bit beyond the screen
ax.set_ylim(-3 * delta_z_total, 3 * delta_z_total) # Sufficient vertical range for both beams
ax.set_xlabel('Horizontal Position (m)')
ax.set_ylabel('Vertical Position (m)')
ax.set_title('Stern-Gerlach Experiment Simulation')
ax.grid(True)

# Draw the "oven" (atom source)
ax.add_patch(plt.Rectangle((-0.1, -0.01), 0.1, 0.02, color='gray', alpha=0.5, label='Oven'))

# Draw the magnet
ax.add_patch(plt.Rectangle((0, -0.02), L, 0.04, color='lightblue', alpha=0.3, label='Magnet'))
ax.add_patch(plt.Polygon([[0, 0.02], [L, 0.02], [L, 0.04], [0, 0.04]], closed=True, color='brown', label='North Pole of Magnet')) # Upper pole
ax.add_patch(plt.Polygon([[0, -0.02], [L, -0.02], [L, -0.04], [0, -0.04]], closed=True, color='darkblue', label='South Pole of Magnet')) # Lower pole


# Draw the screen
ax.axvline(x=L + D, color='green', linestyle='--', label='Screen')

# Initialize trajectory lines
line_up, = ax.plot([], [], 'r-', lw=2, label='Spin Up')
line_down, = ax.plot([], [], 'b-', lw=2, label='Spin Down')

# Points representing moving atoms
atom_up, = ax.plot([], [], 'ro', markersize=8)
atom_down, = ax.plot([], [], 'bo', markersize=8)

# Full trajectories for reference (drawn in background)
x_up_full, z_up_full = get_trajectory(1)
x_down_full, z_down_full = get_trajectory(-1)
ax.plot(x_up_full, z_up_full, 'r--', alpha=0.3)
ax.plot(x_down_full, z_down_full, 'b--', alpha=0.3)

ax.legend()

# --- Initialization function for the animation ---
def init():
    line_up.set_data([], [])
    line_down.set_data([], [])
    atom_up.set_data([], [])
    atom_down.set_data([], [])
    return line_up, line_down, atom_up, atom_down

# --- Update function for each animation frame ---
def update(frame):
    # Calculate trajectories up to the current frame
    x_up, z_up = get_trajectory(1, num_points=frame + 1)
    x_down, z_down = get_trajectory(-1, num_points=frame + 1)

    # Update trajectory lines
    line_up.set_data(x_up, z_up)
    line_down.set_data(x_down, z_down)

    # Update the position of the "atoms"
    if len(x_up) > 0:
        atom_up.set_data([x_up[-1]], [z_up[-1]])
    if len(x_down) > 0:
        atom_down.set_data([x_down[-1]], [z_down[-1]])

    return line_up, line_down, atom_up, atom_down

# Create the animation
num_frames = 200 # Number of frames for the animation
animation = FuncAnimation(fig, update, frames=num_frames, init_func=init, blit=True, interval=50)

# Display the animation
plt.close(fig) # Prevent the static plot from being shown below the animation
HTML(animation.to_jshtml())


# If you want to save the animation as a gif or mp4, you will need ffmpeg or imagemagick
animation.save('stern_gerlach.gif', writer='imagemagick', fps=20)
animation.save('stern_gerlach.mp4', writer='ffmpeg', fps=20)


Scientific Analysis: Spatial Distribution on the Detection Screen

To provide a more quantitative analysis, let's simulate the final vertical positions of many silver atoms on the detection screen. We will consider a large number of atoms, each with either a 'spin up' or 'spin down' orientation, and plot a histogram of their arrival positions. This will illustrate the distinct separation of the two beams as predicted by the Stern-Gerlach experiment.

In [ ]:
# --- Function to calculate final z-position for a single atom ---
def calculate_final_z_position(spin_direction, current_v):
    # Time to traverse the magnet
    t_L = L / current_v
    # Time to travel from magnet to screen
    t_D = D / current_v

    # Vertical displacement upon exiting the magnet
    z_at_exit = 0.5 * (spin_direction * a_z) * t_L**2

    # Vertical velocity upon exiting the magnet
    v_z_at_exit = (spin_direction * a_z) * t_L

    # Final vertical position on the screen
    final_z = z_at_exit + v_z_at_exit * t_D
    return final_z

# --- Simulation of many atoms with velocity distribution ---
num_atoms_total = 10000 # Total number of atoms to simulate

# Maxwell-Boltzmann distribution for an effusive beam (density-weighted, proportional to v^3 * exp(-mv^2/2kBT))
# We can sample velocities directly. The average velocity 'v' we calculated is sqrt(4k_B T / m), which is related to the distribution's peak.
# For an effusive beam, the velocity distribution has a v^3 dependence.
# A simpler way to get a distribution is to use a distribution around the average velocity.

# Let's generate velocities from a Maxwell-Boltzmann distribution (most probable velocity sqrt(2kBT/m))
# For an effusive beam, the *flux* is proportional to v^3 * exp(-mv^2/2kBT).
# However, the problem statement uses v=sqrt(4kBT/m), which is the average for the *effusive beam*.
# To get a distribution around this average, we can approximate a distribution using a scaled chi-distribution or a gamma distribution.
# Or, for simplicity and to show spread, we can use a normal distribution centered at 'v' with some standard deviation.
# A more rigorous approach would be to sample from the correct 1D Maxwell-Boltzmann distribution for beam velocity components.
# For now, let's use a simplified approach to demonstrate spread.

# We will use the 'v' already calculated as a characteristic velocity
# And introduce a spread, e.g., 10-20% of 'v' as a standard deviation for a Gaussian-like distribution.
# A more correct approach for effusive beam is to sample from a distribution proportional to v^3 exp(-mv^2/2kBT).
# For pedagogical purposes to just show spread, let's make a simple velocity distribution.

# Let's generate velocities around the 'v' calculated, using a gamma distribution which is more physical for speeds.
# The mean of a Gamma distribution with shape k and scale theta is k*theta.
# For Maxwell-Boltzmann, the velocity distribution is often related to a Chi distribution.
# For an effusive beam (atoms exiting an oven), the distribution of speeds is f(v) ~ v^3 * exp(-mv^2 / (2kT)).
# Let's use a chi-distribution with 3 degrees of freedom, scaled by sqrt(kT/m).

# A simple approach for a spread: sample velocities from a normal distribution around the mean 'v' with some std_dev.
# Ensure velocities are positive.
# Or, even simpler, consider a small range of velocities around 'v'.

# Let's use a more physically motivated sampling for the effusive beam velocity distribution:
# The distribution of speeds for an effusive beam is P(v) dv = C * v^3 * exp(-mv^2 / (2kT)) dv
# We need to sample from this distribution. Using inverse transform sampling or rejection sampling would be complex.
# For a demonstration, let's approximate by varying velocity within a range.

velocities = np.random.normal(loc=v, scale=v*0.1, size=num_atoms_total) # Gaussian spread around 'v'
velocities = velocities[velocities > 0] # Ensure positive velocities

# Ensure we have enough velocities after filtering
if len(velocities) < num_atoms_total:
    print(f"Warning: Not enough positive velocities generated. Regenerating more...")
    while len(velocities) < num_atoms_total:
        new_velocities = np.random.normal(loc=v, scale=v*0.1, size=num_atoms_total)
        velocities = np.concatenate((velocities, new_velocities[new_velocities > 0]))
    velocities = velocities[:num_atoms_total] # Trim to desired number


# Assign spins randomly (half up, half down)
spin_directions = np.random.choice([1, -1], size=num_atoms_total)

final_z_positions = []
for i in range(num_atoms_total):
    final_z_positions.append(calculate_final_z_position(spin_directions[i], velocities[i]))

final_z_positions = np.array(final_z_positions)

# Separate into spin up and spin down for plotting
spin_up_final_z = final_z_positions[spin_directions == 1]
spin_down_final_z = final_z_positions[spin_directions == -1]

# --- Plotting the distribution on the screen ---
plt.figure(figsize=(10, 7))

# Create a histogram of the final positions
plt.hist(spin_up_final_z, bins=50, alpha=0.6, color='red', label='Spin Up Beam')
plt.hist(spin_down_final_z, bins=50, alpha=0.6, color='blue', label='Spin Down Beam')

# Add a line for the expected separation distance (d) as calculated by the user
plt.axvline(x=delta_z_total, color='red', linestyle='--', label=f'Expected Up Center (Δz_t = {delta_z_total*1000:.3f} mm)')
plt.axvline(x=-delta_z_total, color='blue', linestyle='--', label=f'Expected Down Center (-Δz_t = {-delta_z_total*1000:.3f} mm)')

plt.xlabel('Vertical Position on Screen (m)')
plt.ylabel('Number of Atoms')
plt.title('Distribution of Silver Atoms on the Detection Screen (Stern-Gerlach)')
plt.legend()
plt.grid(True)
plt.show()

print(f"\nSimulated positions for spin up beam: Min={np.min(spin_up_final_z)*1000:.3f} mm, Max={np.max(spin_up_final_z)*1000:.3f} mm")
print(f"Simulated positions for spin down beam: Min={np.min(spin_down_final_z)*1000:.3f} mm, Max={np.max(spin_down_final_z)*1000:.3f} mm")
print(f"Expected half-separation (Δz_t): {delta_z_total*1000:.3f} mm")
print(f"Expected total separation (d): {separation_distance_calculated*1000:.3f} mm")

## 3D Spin Operator Projection and Spinors (Bloch Sphere Representation)


To solve this as a straightforward eigenvalue problem, we first project the spin operator $\mathbf{S}$ onto the unit vector $\mathbf{n}$ and then solve the characteristic equation without invoking the rotation group $SO(3)$ or its double cover $SU(2)$.
The unit vector $\mathbf{n}$ in spherical coordinates is given by:
$\mathbf{n} = \begin{pmatrix} \sin(\beta) \cos(\alpha) \\ \sin(\beta) \sin(\alpha) \\ \cos(\beta) \end{pmatrix}$,
The operator $\mathbf{S} \cdot \mathbf{n}$ is constructed using the components $S_x, S_y, S_z$:
$\mathbf{S} \cdot \mathbf{n} = \begin{pmatrix} S_x \\ S_y \\ S_z \end{pmatrix} \cdot \begin{pmatrix} \sin(\beta) \cos(\alpha) \\ \sin(\beta) \sin(\alpha) \\ \cos(\beta) \end{pmatrix} = S_x \sin(\beta) \cos(\alpha) + S_y \sin(\beta) \sin(\alpha) + S_z \cos(\beta)$,
Instead of a matrix, we express the operator $\mathbf{S} \cdot \mathbf{n}$ using the ladder operators $S_\pm=S_x\pm iS_y$:
$\mathbf{S} \cdot \mathbf{n} = S_z \cos(\beta) + \frac{1}{2} \sin(\beta) (S_+ e^{-i\alpha} + S_- e^{i\alpha})$,
Let $|\mathbf{n}; +\rangle = a |+\rangle + b|-\rangle$. The eigenvalue equation is:
$[S_z \cos(\beta) + \frac{1}{2} \sin(\beta) (S_+ e^{-i\alpha} + S_- e^{i\alpha})](a |+\rangle + b|-\rangle) = \frac{\hbar}{2} (a |+\rangle + b|-\rangle)$,


Applying the operators:
$S_z |+\rangle = \frac{\hbar}{2}|+\rangle, S_z |-\rangle = -\frac{\hbar}{2}|-\rangle$,
$S_+ |-\rangle = \hbar|+\rangle, S_+ |+\rangle = 0$,
$S_- |+\rangle = \hbar|-\rangle, S_- |-\rangle = 0$,
Substitute these into the equation and project onto $\langle+| $ and $\langle-|$:
Projection onto $\langle+|$:
$a \frac{\hbar}{2} \cos(\beta) + b \frac{\hbar}{2} \sin(\beta) e^{-i\alpha} = \frac{\hbar}{2} a$,
$\therefore b/a = \frac{1-\cos(\beta)}{\sin(\beta)} e^{i\alpha}$,
Projection onto $\langle-|$:
$-b \frac{\hbar}{2} \cos(\beta) + a \frac{\hbar}{2} \sin(\beta) e^{i\alpha} = \frac{\hbar}{2} b$,
$\therefore b/a = \frac{\sin(\beta)}{1+\cos(\beta)} e^{i\alpha}$,
Using the identity $\tan(\beta/2) = \frac{1-\cos(\beta)}{\sin(\beta)} = \frac{\sin(\beta)}{1+\cos(\beta)}$, both projections yield:
$b/a = \tan(\beta/2) e^{i\alpha}$,
By setting $a=\cos(\beta/2)$ to satisfy $\langle\Psi|\Psi\rangle=1$:
$\therefore |\mathbf{n}; +\rangle = \cos(\beta/2) |+\rangle + \sin(\beta/2) e^{i\alpha} |-\rangle$.
Notice that even though the physical direction $\mathbf{n}$ is periodic in $2\pi$ for $\beta$, the state vector only returns to its original value after a $4\pi$ rotation (due to the $\beta/2$ factor). This is the fundamental definition of a spinor and illustrates why $SU(2)$ is the universal cover of $SO(3)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function to draw a Bloch sphere
def plot_bloch_sphere(ax, title="Bloch Sphere Representation of a Spin State"):
    # Draw the sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x, y, z, color='lightgray', alpha=0.1, rstride=5, cstride=5)

    # Draw axes
    ax.plot([-1.1, 1.1], [0, 0], [0, 0], 'k--')
    ax.plot([0, 0], [-1.1, 1.1], [0, 0], 'k--')
    ax.plot([0, 0], [0, 0], [-1.1, 1.1], 'k--')

    # Labels for axes
    ax.text(0, 0, 1.2, '|↑⟩', color='blue') # Spin up
    ax.text(0, 0, -1.2, '|↓⟩', color='red') # Spin down
    ax.text(1.2, 0, 0, '|→⟩', color='black') # Spin right (for x-direction)
    ax.text(-1.2, 0, 0, '|←⟩', color='black') # Spin left
    ax.text(0, 1.2, 0, '|↻⟩', color='black') # Spin +y
    ax.text(0, -1.2, 0, '|↺⟩', color='black') # Spin -y

    ax.set_box_aspect([1,1,1]) # Equal aspect ratio
    ax.set_axis_off()
    ax.set_title(title)

# Example: Plot a general spin state vector on the Bloch sphere
# Let's pick some arbitrary angles for demonstration, corresponding to a general state
beta_example = np.pi / 4  # Polar angle (from z-axis)
alpha_example = np.pi / 3  # Azimuthal angle (in xy-plane from x-axis)

# Coordinates of the vector on the Bloch sphere
x_vec = np.sin(beta_example) * np.cos(alpha_example)
y_vec = np.sin(beta_example) * np.sin(alpha_example)
z_vec = np.cos(beta_example)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

plot_bloch_sphere(ax)

# Plot the spin state vector
ax.quiver(0, 0, 0, x_vec, y_vec, z_vec, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3, label='|n̂;+⟩')

# Add the angles for clarity
ax.plot([0, x_vec], [0, 0], [0, 0], 'r--', alpha=0.5)
ax.plot([0, 0], [0, y_vec], [0, 0], 'r--', alpha=0.5)
ax.plot([0, x_vec], [0, y_vec], [0, 0], 'k:', alpha=0.5) # Projection in xy-plane

ax.text(x_vec*1.1, y_vec*1.1, z_vec*1.1, rf'$\beta={beta_example/np.pi:.2f}\pi$, $\alpha={alpha_example/np.pi:.2f}\pi$', color='darkgreen')

ax.legend()
plt.show()


## Temporal Evolution and Angle Variation on the Bloch Sphere (Animated)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Re-using the plot_bloch_sphere function for the static elements
def plot_bloch_sphere_base(ax, title="Bloch Sphere with Precessing Spin State"):
    # Draw the sphere
    u = np.linspace(0, 2 * np.pi, 100)
    v = np.linspace(0, np.pi, 100)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x, y, z, color='lightgray', alpha=0.1, rstride=5, cstride=5)

    # Draw axes
    ax.plot([-1.1, 1.1], [0, 0], [0, 0], 'k--')
    ax.plot([0, 0], [-1.1, 1.1], [0, 0], 'k--')
    ax.plot([0, 0], [0, 0], [-1.1, 1.1], 'k--')

    # Labels for axes
    ax.text(0, 0, 1.2, '|↑⟩', color='blue') # Spin up
    ax.text(0, 0, -1.2, '|↓⟩', color='red') # Spin down
    ax.text(1.2, 0, 0, '|→⟩', color='black') # Spin right (for x-direction)
    ax.text(-1.2, 0, 0, '|←⟩', color='black') # Spin left
    ax.text(0, 1.2, 0, '|↻⟩', color='black') # Spin +y
    ax.text(0, -1.2, 0, '|↺⟩', color='black') # Spin -y

    ax.set_box_aspect([1,1,1]) # Equal aspect ratio
    ax.set_axis_off()
    ax.set_title(title)

# --- Animation Setup ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

plot_bloch_sphere_base(ax)

# Initial state parameters
initial_beta = np.pi / 3  # Start at some polar angle
initial_alpha = np.pi / 4  # Start at some azimuthal angle

# Create the vector for the spin state
line, = ax.plot([], [], [], 'darkgreen', marker='o', markersize=8, markeredgecolor='darkgreen', label='Spin State Vector')
quiver = ax.quiver(0, 0, 0, 0, 0, 0, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3)

# Text to display angles
angle_text = ax.text2D(0.05, 0.95, '', transform=ax.transAxes, color='darkgreen', fontsize=12)

# Function to initialize the animation
def init_animation():
    global quiver
    x_vec = np.sin(initial_beta) * np.cos(initial_alpha)
    y_vec = np.sin(initial_beta) * np.sin(initial_alpha)
    z_vec = np.cos(initial_beta)

    # Remove previous quiver if it exists
    if 'quiver' in globals() and quiver is not None:
        quiver.remove()

    quiver = ax.quiver(0, 0, 0, x_vec, y_vec, z_vec, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3)
    angle_text.set_text(rf'$\beta_0={initial_beta/np.pi:.2f}\pi$, $\alpha_0={initial_alpha/np.pi:.2f}\pi$')
    return quiver, angle_text

# Function to update for each frame of the animation
def update_animation(frame):
    global quiver
    # Simulate precession (alpha changes linearly, beta oscillates slightly)
    time = frame * 0.05 # Time step

    # Azimuthal angle changes linearly with time (precession)
    current_alpha = (initial_alpha + time * np.pi/2) % (2 * np.pi)

    # Polar angle oscillates slightly around the initial_beta
    current_beta = initial_beta + 0.1 * np.sin(time * 0.5)

    # Ensure beta stays within [0, pi]
    current_beta = np.clip(current_beta, 0, np.pi)

    x_vec = np.sin(current_beta) * np.cos(current_alpha)
    y_vec = np.sin(current_beta) * np.sin(current_alpha)
    z_vec = np.cos(current_beta)

    # Remove old quiver and draw new one
    if quiver is not None:
        quiver.remove()
    quiver = ax.quiver(0, 0, 0, x_vec, y_vec, z_vec, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3)

    angle_text.set_text(rf'$\beta={current_beta/np.pi:.2f}\pi$, $\alpha={current_alpha/np.pi:.2f}\pi$')

    # Set view to follow the vector slightly for better visualization
    ax.view_init(elev=30 + 10*np.sin(time*0.3), azim=time*50)

    return quiver, angle_text

# Create the animation
num_frames_animation = 100 # Number of frames
ani = FuncAnimation(fig, update_animation, frames=num_frames_animation, init_func=init_animation, blit=False, interval=100)

# Display the animation in the notebook
plt.close(fig) # Prevent static plot from showing
html_ani = HTML(ani.to_jshtml())
display(html_ani)

# Save the animation as a GIF
print("Saving animation as GIF...")
ani.save('bloch_sphere_temporal_evolution.gif', writer='pillow', fps=10)
print("GIF saved: bloch_sphere_temporal_evolution.gif")

## Heisenberg Uncertainty Principle and the Ice Pick: A Quantum Stability Simulation

### Problem Statement
Estimate the rough order of magnitude of the length of time that an ice pick can be balanced on its point if the only limitation is that set by the Heisenberg uncertainty principle. Assume that the point is sharp and that the point and the surface on which it rests are hard. You may make approximations which do not alter the general order of magnitude of the result. Assume reasonable values for the dimensions and weight of the ice pick. Obtain an approximate numerical result and express it in seconds.

### Analytical Solution
We model the ice pick as a rigid rod of length $L$ and mass $m$, balanced on a sharp point. Let $\theta$ be the angle from the vertical. For small angles, the equation of motion is:

$I\ddot{\theta} = \tau = mgh \sin(\theta) \approx mgh\theta$,

Where $I$ is the moment of inertia about the tip and $h$ is the distance from the tip to the center of mass. Defining $\omega^2 = mgh/I$, the general solution is:

$\theta(t) = \theta_0 \cosh(\omega t) + \frac{\dot{\theta}_0}{\omega} \sinh(\omega t) \approx \frac{1}{2} (\theta_0 + \frac{\dot{\theta}_0}{\omega}) e^{\omega t}$,

where we consider only the growing exponential term for large $t$.
According to the Heisenberg Uncertainty Principle, the initial position and momentum cannot both be zero. In angular variables:

$\Delta\theta \Delta L_\theta \ge \hbar/2 \implies \theta_0 (I\dot{\theta}_0) \sim \hbar/2$,

To maximize the time $t$, we must minimize the initial "kick" coefficient $A = \theta_0 + \frac{\dot{\theta}_0}{\omega}$.
Substituting $\dot{\theta}_0 = \frac{\hbar}{2I\theta_0}$:

$A(\theta_0) = \theta_0 + \frac{\hbar}{2I\omega\theta_0}$,

Minimizing $A$ with respect to $\theta_0$ ($\frac{dA}{d\theta_0} = 0$) gives:

$1 - \frac{\hbar}{2I\omega\theta_0^2} = 0$,

$\theta_0 = \sqrt{\frac{\hbar}{2I\omega}} \implies A_{min} = \sqrt{\frac{2\hbar}{I\omega}}$,

We assume reasonable values for an ice pick:
*   Length ($L$): $\approx 15\text{cm} = 0.15\text{m}$.
*   Mass($m$): $\approx 100\text{g} = 0.1\text{Kg}$.
*   Geometry: Assuming a uniform rod, $h=L/2=0.075\text{m}$ and $I=\frac{1}{3} mL^2=7.5\times10^{-4}\text{ Kg} \cdot\text{m}^2$.
*   Characteristic frequency: $\omega=\sqrt{\frac{3g}{2L}}=\sqrt{\frac{3(9.8)}{0.3}} \text{ s}^{-1}\approx 9.9\text{s}^{-1}$.

The ice pick "falls" when the angle reaches the order of $\theta(t)\sim 1$ radian:

$1 \approx \sqrt{\frac{2\hbar}{I\omega}} e^{\omega t} \implies t \approx \frac{1}{2\omega} \ln\left(\frac{I\omega}{2\hbar}\right)$,

Plugging in the constants ($\hbar \approx 1.054\times10^{-34}\text{ J}\cdot\text{s}$):

$t \approx \frac{1}{2(9.9\text{s}^{-1})} \ln\left(\frac{(7.5\times10^{-4}\text{ Kg} \cdot\text{m}^2)(9.9\text{s}^{-1})}{2(1.054\times10^{-34}\text{ J}\cdot\text{s})}\right)$,

$t \approx \frac{1}{19.8\text{s}^{-1}} \ln(3.5\times10^{31}) \approx \frac{72.6}{19.8} \text{ s} \approx 3.7\text{s}$.

The rough order of magnitude for balancing an ice pick under purely quantum limitations is approximately 3 to 4 seconds. Despite the tiny scale of $\hbar$, the exponential growth of the instability ($e^{\omega t}$) amplifies quantum uncertainties to macroscopic scales very rapidly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Physical constants and ice pick parameters ---
L = 0.15      # Length of the ice pick in meters
m = 0.1       # Mass of the ice pick in kg
g = 9.8       # Acceleration due to gravity in m/s^2
hbar = 1.054e-34 # Reduced Planck constant in J s

# Derived parameters
h = L / 2     # Distance from tip to center of mass (for uniform rod)
I = (1/3) * m * L**2 # Moment of inertia about the tip (for uniform rod)
omega = np.sqrt((3 * g) / (2 * L)) # Characteristic frequency, using I = (1/3)mL^2 and h = L/2

print(f"Ice pick length (L): {L} m")
print(f"Ice pick mass (m): {m} kg")
print(f"Moment of inertia (I): {I:.2e} kg m^2")
print(f"Characteristic frequency (ω): {omega:.2f} s^-1")
print(f"Reduced Planck constant (ħ): {hbar:.2e} J s")

# --- Monte Carlo Simulation ---
num_simulations = 50000 # Number of ice pick balancing attempts

# Optimal initial angle (theta_0) that maximizes balancing time
theta_0_optimal = np.sqrt(hbar / (2 * I * omega))

# Sample initial angles (theta_0) around the optimal value
# We use a log-normal distribution to ensure positive values and cover a reasonable range,
# centered around the optimal theta_0, but with some spread.
# Alternatively, a simple normal distribution with filtering for small positive values is also possible.
# For pedagogical reasons, let's use a normal distribution with clipping.

sd_factor = 0.5 # Standard deviation as a fraction of the optimal theta_0
theta_0_samples = np.random.normal(loc=theta_0_optimal, scale=theta_0_optimal * sd_factor, size=num_simulations)

# Ensure initial angles are positive and within a physically relevant small range
theta_0_samples = np.clip(theta_0_samples, 1e-20, 1e-10) # Clamp to very small but non-zero values

# Calculate corresponding initial angular velocities (theta_dot_0) from Uncertainty Principle
# theta_0 * I * theta_dot_0 = hbar / 2
theta_dot_0_samples = hbar / (2 * I * theta_0_samples)

# Calculate the 'kick' coefficient A for each sample
A_samples = theta_0_samples + theta_dot_0_samples / omega

# Calculate the balancing time 't' for each sample
# t = (1/omega) * ln(2 / A)
# Handle cases where A might be too large (e.g., if theta_0 is too small and theta_dot_0 becomes very large)
# If A >= 2, ln(2/A) would be <= 0, implying instant fall or no growth.
# In such cases, the approximation might break down, or time is essentially zero.
# We will filter out invalid times later or set them to a very small positive value.

t_samples = np.zeros(num_simulations)
for i in range(num_simulations):
    if A_samples[i] > 0 and 2 / A_samples[i] > 1: # Ensure argument to log is > 1 for positive time
        t_samples[i] = (1 / omega) * np.log(2 / A_samples[i])
    else:
        t_samples[i] = 0.0 # Effectively immediate fall if A is too large or invalid

# Filter out any non-positive times if necessary (though the above clip handles it)
t_samples = t_samples[t_samples > 0]

# --- Visualization ---
plt.figure(figsize=(10, 6))
plt.hist(t_samples, bins=50, density=True, alpha=0.7, color='purple', edgecolor='black')
plt.title('Distribution of Ice Pick Balancing Times due to Quantum Uncertainty')
plt.xlabel('Balancing Time (seconds)')
plt.ylabel('Probability Density')
plt.grid(True)

# Add vertical line for the analytical solution's time
t_analytical = (1 / (2 * omega)) * np.log((I * omega) / (2 * hbar))
plt.axvline(t_analytical, color='red', linestyle='dashed', linewidth=2, label=f'Analytical (optimal) Time: {t_analytical:.2f} s')
plt.legend()
plt.show()

print(f"\nSimulated average balancing time: {np.mean(t_samples):.2f} s")
print(f"Simulated median balancing time: {np.median(t_samples):.2f} s")
print(f"Simulated max balancing time: {np.max(t_samples):.2f} s")
print(f"Simulated min balancing time: {np.min(t_samples):.2e} s")
print(f"Analytical optimal balancing time: {t_analytical:.2f} s")


### Visualizing the Ice Pick's Temporal Evolution

To provide a clearer understanding of the ice pick's behavior under quantum uncertainty, we will animate its temporal evolution. Starting from the minute initial quantum-induced angular displacement and velocity, the ice pick will gradually lean further and further from the vertical until it topples over. This visualization will show how small quantum fluctuations are amplified into a macroscopic fall.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Re-use constants and derived parameters from the previous cell
# L, m, g, hbar, I, omega, t_analytical, t_samples are assumed to be defined in previous cells

# If you run this cell independently, ensure these are defined:
# L = 0.15      # Length of the ice pick in meters
# m = 0.1       # Mass of the ice pick in kg
# g = 9.8       # Acceleration due to gravity in m/s^2
# hbar = 1.054e-34 # Reduced Planck constant in J s
# I = (1/3) * m * L**2 # Moment of inertia about the tip (for uniform rod)
# omega = np.sqrt((3 * g) / (2 * L)) # Characteristic frequency
# t_analytical = (1 / (2 * omega)) * np.log((I * omega) / (2 * hbar)) # Analytical optimal time
# t_samples = ... # Array of balancing times from Monte Carlo simulation

# Initial conditions for animation, adjusted for better visualization
# Using the optimal theta_0 that maximizes balancing time, for a longer fall as requested.
theta_0_optimal = np.sqrt(hbar / (2 * I * omega))
theta_0_anim = theta_0_optimal # Set initial angle for animation to optimal value

# Calculate corresponding initial angular velocity from Uncertainty Principle
theta_dot_0_anim = hbar / (2 * I * theta_0_anim)

print(f"Initial angle for animation (theta_0): {theta_0_anim:.2e} radians")
print(f"Initial angular velocity for animation (theta_dot_0): {theta_dot_0_anim:.2e} rad/s")

# Time evolution
# Calculate the time it takes for *this* specific animated ice pick to fall (theta_t_anim >= 1 radian)
# This will now be close to t_analytical

# Calculate the 'kick' coefficient A_anim using the chosen initial conditions
A_anim = theta_0_anim + (theta_dot_0_anim / omega)

# Determine t_fall_this_anim (when theta reaches 1 radian) more precisely
# We'll run a temporary simulation to find this exact fall time
t_max_anim_initial_guess = t_analytical * 1.5 # Start with a generous upper bound for finding fall time
t_values_temp = np.linspace(0, t_max_anim_initial_guess, 500) # Temporary time values

theta_t_temp = np.zeros_like(t_values_temp)
for i, t in enumerate(t_values_temp):
    theta_t_temp[i] = 0.5 * A_anim * np.cosh(omega * t) + 0.5 * (theta_0_anim - (theta_dot_0_anim / omega)) * np.sinh(omega * t)

t_fall_idx = np.where(theta_t_temp >= 1)[0]
if len(t_fall_idx) > 0:
    t_fall_this_anim = t_values_temp[t_fall_idx[0]]
else:
    t_fall_this_anim = t_max_anim_initial_guess # Fall not reached within guess, use max guess

# Now set t_max_anim based on the calculated fall time, slightly extending to show complete fall
t_max_anim = t_fall_this_anim * 1.05 # User requested ~3.7-3.8s, this will be around 3.85s if t_analytical is ~3.67s
num_frames_anim = int(t_max_anim * 20 * 2) # Adjust frames based on duration for smoother animation (e.g., 40 fps)
if num_frames_anim < 100: num_frames_anim = 100 # Ensure minimum frames for smooth animation

t_values_anim = np.linspace(0, t_max_anim, num_frames_anim)

# Recalculate theta_t_anim with the final t_values_anim
theta_t_anim = np.zeros_like(t_values_anim)
for i, t in enumerate(t_values_anim):
    theta_t_anim[i] = 0.5 * A_anim * np.cosh(omega * t) + 0.5 * (theta_0_anim - (theta_dot_0_anim / omega)) * np.sinh(omega * t)


print(f"Calculated fall time for this animation: {t_fall_this_anim:.3f} s")
print(f"Total animation duration: {t_max_anim:.3f} s")
print(f"Number of animation frames: {num_frames_anim}")

# --- Animation Setup ---
# Adjusted figsize for a 2-row layout
fig = plt.figure(figsize=(15, 10)) # Wider for 2 columns, taller for 2 rows
# Changed GridSpec to 2 rows, 2 columns
grid = plt.GridSpec(2, 2, height_ratios=[2, 1]) # 2 rows, 2 columns, top row taller

ax_main = fig.add_subplot(grid[0, 0]) # Main Ice Pick View (top-left)
ax_magnified = fig.add_subplot(grid[0, 1]) # Magnified View (top-right)
ax_hist = fig.add_subplot(grid[1, :]) # Histogram (bottom row, spanning both columns)

# --- Main Ice Pick View (ax_main) ---
ax_main.set_xlim(-L * 1.5, L * 1.5)
ax_main.set_ylim(-L * 0.5, L * 1.5)
ax_main.set_aspect('equal', adjustable='box')
ax_main.set_title(f'Quantum-Limited Ice Pick Fall (Initial Angle = {theta_0_anim:.2e} rad)') # Update title to reflect new angle
ax_main.grid(True) # Restore grid
ax_main.set_xlabel('Horizontal Position (m)') # Restore labels
ax_main.set_ylabel('Vertical Position (m)') # Restore labels
ax_main.set_axis_on() # Restore axes
ax_main.plot(0, 0, 'ko', markersize=8, label='Pivot Point')
line_pick_main, = ax_main.plot([], [], 'deepskyblue', lw=10, label='Ice Pick')
angle_time_text = ax_main.text(0.05, 0.9, '', transform=ax_main.transAxes, fontsize=12, color='darkred') # Angle and time text

# --- Magnified View (ax_magnified) ---
# Set fixed, small limits to ensure a visible plot, even with tiny initial angle
mag_fixed_range = 1e-5 # A small, visually sensible range for the magnified view
ax_magnified.set_xlim(-mag_fixed_range, mag_fixed_range)
ax_magnified.set_ylim(L - mag_fixed_range, L + mag_fixed_range)
ax_magnified.set_aspect('equal', adjustable='box')
ax_magnified.set_title('Magnified Initial Wobble')
ax_magnified.set_xlabel('x-position (m)')
ax_magnified.set_ylabel('y-position (m)')
ax_magnified.grid(True)
ax_magnified.plot(0, L, 'kx', markersize=10, label='Upright Position') # Ideal upright top
point_magnified, = ax_magnified.plot([], [], 'red', marker='o', markersize=10, label='Ice Pick Top')

# --- Monte Carlo Histogram View (ax_hist) ---
ax_hist.hist(t_samples, bins=50, density=True, alpha=0.7, color='purple', edgecolor='black')
ax_hist.axvline(t_analytical, color='red', linestyle='--', linewidth=2, label=f'Analytical (optimal) Time: {t_analytical:.2f} s')
ax_hist.set_title('Distribution of Balancing Times (Monte Carlo)')
ax_hist.set_xlabel('Balancing Time (seconds)')
ax_hist.set_ylabel('Probability Density')
ax_hist.grid(True)
ax_hist.legend()

# Initialize a marker for the current animation time on the histogram
hist_time_marker, = ax_hist.plot([], [], 'blue', linestyle='-', lw=3, label='Current Animation Time') # Keep blue as requested
ax_hist.legend()

plt.tight_layout() # Adjust layout to prevent overlapping

# --- Initialization function for the animation ---
def init_ice_pick_anim():
    line_pick_main.set_data([], [])
    point_magnified.set_data([], [])
    angle_time_text.set_text('') # Initialize angle/time text
    hist_time_marker.set_data([], [])
    return line_pick_main, point_magnified, angle_time_text, hist_time_marker # Adjusted return tuple

# --- Update function for each animation frame ---
def update_ice_pick_anim(frame):
    current_theta = theta_t_anim[frame]
    current_time = t_values_anim[frame]

    # Calculate coordinates for the ice pick's top
    x_tip = 0
    y_tip = 0
    x_top = x_tip + L * np.sin(current_theta)
    y_top = y_tip + L * np.cos(current_theta)

    # Update main view
    line_pick_main.set_data([x_tip, x_top], [y_tip, y_top])
    angle_time_text.set_text(f'Angle: {np.degrees(current_theta):.2f}°\nTime: {current_time:.2f} s') # Update angle/time text

    # Update magnified view
    point_magnified.set_data([x_top], [y_top]) # Top point

    # Update histogram time marker
    if current_time <= t_fall_this_anim:
        hist_time_marker.set_data([current_time, current_time], [0, ax_hist.get_ylim()[1]])
        hist_time_marker.set_color('blue') # Keep blue as requested
    else:
        # After fall time, keep the marker at fall_time and change color to red
        hist_time_marker.set_data([t_fall_this_anim, t_fall_this_anim], [0, ax_hist.get_ylim()[1]])
        hist_time_marker.set_color('red')

    return line_pick_main, point_magnified, angle_time_text, hist_time_marker # Adjusted return tuple

# Create the animation
ani_ice_pick = FuncAnimation(fig, update_ice_pick_anim, frames=num_frames_anim,
                             init_func=init_ice_pick_anim, blit=True, interval=50)

# Display the animation
plt.close(fig)
html_ani_ice_pick = HTML(ani_ice_pick.to_jshtml())
display(html_ani_ice_pick)

# Save the animation as a GIF
print("Saving custom ice pick animation as GIF...")
ani_ice_pick.save('ice_pick_fall_custom.gif', writer='pillow', fps=20)
print("GIF saved: ice_pick_fall_custom.gif")

## Fourier Transforms in Quantum Mechanics: Position and Momentum Operators

In quantum mechanics, the position and momentum representations are deeply connected through Fourier transforms. This section explores this relationship, particularly how operators and wavefunctions transform between these two fundamental bases.

### Continuum Analogue for Position and Momentum

As derived, in the continuum limit, the resolution of identity in the position basis is $I=\int |x\rangle\langle x| d^3 x$. For a function $F(r)$ of the position operator $\hat{x}$, it is diagonal in the position representation: $F(r)|x\rangle=F(|x|)|x\rangle$. The matrix element in the momentum basis is given by:

$\langle p''|F(r)|p' \rangle = \frac{1}{(2\pi\hbar)^3} \int F(x) e^{-i (p''-p') \cdot x / \hbar} d^3 x$

Let $q = |p''-p'|/\hbar$ be the wave-vector transfer. For a purely radial function $F(r)$, this simplifies to a Fourier-Sine transform:

$\langle p''|F(r)|p' \rangle = \frac{1}{2\pi^2 \hbar^3 q} \int_0^\infty r F(r) \sin(qr)dr$

This result establishes that scattering amplitudes in the momentum representation are governed by the spectral decomposition of the interaction potential. Let's visualize this with an example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define a sample radial function F(r)
def F_r_exp(r):
    """Example radial function: a decaying exponential."""
    return np.exp(-r)

# Analytical Fourier-Sine transform for F(r) = exp(-r)
def fourier_sine_transform_F_r_exp_analytical(q, hbar_val):
    """Calculates the analytical Fourier-Sine transform for F(r) = exp(-r)."""
    # The integral we want is I = int_0^inf r * exp(-r) * sin(qr) dr = 2q / (1 + q^2)^2
    integral_result = 2 * q / ((1 + q**2)**2)

    # The prefactor from the derivation
    prefactor = 1 / (2 * np.pi**2 * hbar_val**3 * q)

    return prefactor * integral_result

# --- Constants ---
hbar_val = 1.054e-34 # Reduced Planck constant (J s)

# --- Calculation ---
# Define a range for momentum transfer q
q_values = np.logspace(-3, 11, 200)

momentum_matrix_elements_exp = [
    fourier_sine_transform_F_r_exp_analytical(q, hbar_val)
    for q in q_values
]

# --- Visualization ---
plt.figure(figsize=(10, 6))
plt.plot(q_values, np.abs(momentum_matrix_elements_exp), label=r'$|\tilde{F}(q)| \propto |\langle p_2|e^{-r}|p_1 \rangle|$')
plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'Momentum Transfer $q = |p_2-p_1|/\hbar$ (arbitrary units)')
plt.ylabel('Magnitude of Matrix Element (Arbitrary Units)')
plt.title('Fourier-Sine Transform of $e^{-r}$ Potential (Momentum Representation)')
plt.grid(True, which="both", ls="--")
plt.legend()
plt.show()

print("Demonstration of Fourier-Sine Transform for a simple decaying exponential F(r).")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the generalized radial function F(r) for Yukawa potential
def F_r_yukawa(r, k_val):
    """Generalized Yukawa potential: F(r) = exp(-k*r) / r."""
    # Add a small epsilon to avoid division by zero at r=0
    epsilon = 1e-9
    return np.exp(-k_val * r) / (r + epsilon)

# Define the analytical Fourier-Sine transform for F_r_yukawa(r, k)
def fourier_sine_transform_F_r_yukawa_analytical(q, k_val, hbar_val):
    """Calculates the analytical Fourier-Sine transform for F(r) = exp(-k*r)/r.
    The integral is int_0^inf r * (exp(-k*r)/r) * sin(qr) dr = int_0^inf exp(-k*r) * sin(qr) dr = q / (k^2 + q^2).
    """
    # Handle the case where q is very small and k_val is also very small (approximating Coulomb)
    # If q is zero, the integral is zero. For non-zero q, we use the formula.
    if q == 0:
        integral_result = 0.0
    else:
        integral_result = q / (k_val**2 + q**2)

    # The prefactor from the derivation
    prefactor = 1 / (2 * np.pi**2 * hbar_val**3 * q)

    return prefactor * integral_result

# --- Constants ---
hbar_val = 1.054e-34 # Reduced Planck constant (J s)

# --- Calculation ---
# Define a range for momentum transfer q
# Avoid q=0 because of the 1/q term in the prefactor
q_values = np.logspace(-3, 11, 200) # Use logspace for better distribution over a wide range

# Define different k values to show the transition from Coulomb to Yukawa
k_values_to_plot = [
    1e-10, # Very small k, approximating Coulomb
    1e-5,  # Small k, long-range Yukawa
    1,     # k=1, as in previous exp(-r) example
    100    # Large k, short-range Yukawa
]

plt.figure(figsize=(10, 6))

for k_val in k_values_to_plot:
    # Calculate momentum matrix elements for each k value
    momentum_matrix_elements = [
        fourier_sine_transform_F_r_yukawa_analytical(q, k_val, hbar_val)
        for q in q_values
    ]
    # Plot the magnitude of the matrix elements
    plt.plot(q_values, np.abs(momentum_matrix_elements), label=f'$k={k_val}$ $(1/m)$')

plt.xscale('log')
plt.yscale('log')
plt.xlabel(r'Momentum Transfer $q = |p_2-p_1|/\hbar$ (arbitrary units)')
plt.ylabel('Magnitude of Matrix Element (Arbitrary Units)')
plt.title('Fourier-Sine Transform of Yukawa Potential (Momentum Representation)')
plt.grid(True, which="both", ls="--")
plt.legend(title='Screening Parameter $k$')
plt.show()

print("Demonstration of Fourier-Sine Transform for a radial function F(r).")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Define the analytical Fourier-Sine transform for F(r) = exp(-k*r)/r
def fourier_sine_transform_F_r_yukawa_analytical(q, k_val, hbar_val):
    """Calculates the analytical Fourier-Sine transform for F(r) = exp(-k*r)/r.
    The integral is int_0^inf r * (exp(-k*r)/r) * sin(qr) dr = int_0^inf exp(-k*r) * sin(qr) dr = q / (k^2 + q^2).
    """
    # Handle the case where q is very small and k_val is also very small (approximating Coulomb)
    # If q is zero, the integral is zero. For non-zero q, we use the formula.
    if q == 0:
        integral_result = 0.0
    else:
        integral_result = q / (k_val**2 + q**2)

    # The prefactor from the derivation
    prefactor = 1 / (2 * np.pi**2 * hbar_val**3 * q)

    return prefactor * integral_result

# --- Constants ---
hbar_val = 1.054e-34 # Reduced Planck constant (J s)

# --- Calculation Setup ---
# Define a range for momentum transfer q
q_values = np.logspace(-3, 11, 200) # Same range as previous plots

# Define the range for the screening parameter k for the animation
k_start = 1e-10 # Approximating Coulomb
k_end = 100     # Strong screening
num_animation_frames = 200
k_animation_values = np.logspace(np.log10(k_start), np.log10(k_end), num_animation_frames)

# --- Animation Setup ---
fig, ax = plt.subplots(figsize=(10, 6))

# Set logarithmic scales for both axes
ax.set_xscale('log')
ax.set_yscale('log')

# Set labels and title
ax.set_xlabel(r'Momentum Transfer $q = |p_2-p_1|/\hbar$ (arbitrary units)')
ax.set_ylabel('Magnitude of Matrix Element (Arbitrary Units)')
ax.set_title('Fourier-Sine Transform of Yukawa Potential (Animation)')
ax.grid(True, which="both", ls="--")

# Initialize the plot line for the Fourier transform
line, = ax.plot([], [], lw=2, color='darkorange')

# Initialize text for displaying k value dynamically
k_text = ax.text(0.95, 0.95, '', transform=ax.transAxes, ha='right', va='top', fontsize=12)

# Set initial axis limits based on the previous plot's full range (min/max of all k_values_to_plot)
# We'll calculate the min/max y-values for the entire k_animation_values range to set fixed limits
all_y_values = []
for k_val in k_animation_values:
    momentum_matrix_elements_k = [
        fourier_sine_transform_F_r_yukawa_analytical(q, k_val, hbar_val)
        for q in q_values
    ]
    all_y_values.extend(np.abs(momentum_matrix_elements_k))

min_y = np.min(all_y_values)
max_y = np.max(all_y_values)

ax.set_xlim(q_values.min(), q_values.max())
ax.set_ylim(min_y * 0.5, max_y * 2) # Add some padding


# --- Animation Functions ---
def init():
    line.set_data([], [])
    k_text.set_text('')
    return line, k_text

def update(frame):
    current_k = k_animation_values[frame]

    # Calculate momentum matrix elements for the current k
    momentum_matrix_elements = [
        fourier_sine_transform_F_r_yukawa_analytical(q, current_k, hbar_val)
        for q in q_values
    ]

    # Update the plot data
    line.set_data(q_values, np.abs(momentum_matrix_elements))

    # Update the k value text in scientific notation
    k_text.set_text(f'$k = {current_k:.1e}$ $(1/m)$')

    return line, k_text

# Create the animation
animation = FuncAnimation(fig, update, frames=num_animation_frames,
                          init_func=init, blit=True, interval=100) # Interval in ms

# Display the animation in the notebook as HTML
plt.close(fig) # Prevent the static plot from being shown below the animation
html_animation = HTML(animation.to_jshtml())
display(html_animation)

# Save the animation as a GIF
print("Saving animation as GIF...")
animation.save('yukawa_fourier_transform_animation.gif', writer='pillow', fps=20)
print("GIF saved: yukawa_fourier_transform_animation.gif")

print("Animation of Fourier-Sine Transform for varying Yukawa potential (k parameter).")

## Spin Rotations and Change of Basis

### 1.26 a. Prove that (1/√2)(1+iσ_x ), where the matrix σ_x is :

$ \sigma_x = \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix} $,

acting on a two-component spinor can be regarded as the matrix representation of the rotation operator about the x-axis by angle −π/2. (The minus sign signifies that the rotation is clockwise.)

Solution:

**a. Unitary Representation of Rotations**
In the context of a spin-1/2 system, the rotation operator for a rotation by an angle $\phi$ about an axis defined by the unit vector $\hat{n}$ is given by the exponential map:

$ D(\hat{n},\phi)=exp(-i (\vec{S}\cdot\hat{n})/\hbar \phi)=exp(-i (\vec{\sigma}\cdot\hat{n})/2 \phi) $

For a rotation about the x-axis ($\hat{n}=\hat{e}_x$), the operator simplifies to:

$ D_x (\phi)=exp(-i \sigma_x/2 \phi) $

Using the fundamental property of Pauli matrices ($\sigma_k^2=I$), the Taylor expansion of the exponential yields the Euler-like identity:

$ exp(i\sigma_k \theta)=I \cos(\theta)+i\sigma_k \sin(\theta) $

Setting the angle $\phi=-\pi/2$, we define the argument $\theta=-\phi/2=\pi/4$. Substituting this into the identity:

$ D_x (-\pi/2)=I \cos(\pi/4)+i\sigma_x \sin(\pi/4) $

Since $ \cos(\pi/4)=\sin(\pi/4)=1/\sqrt{2} $, we obtain:

$ D_x (-\pi/2)=1/\sqrt{2} (I+i\sigma_x ) $

The operator given is precisely the unitary matrix representing a clockwise rotation of $\pi/2$ radians in the spinor space.

In [ ]:
import numpy as np

# Define Pauli matrices and Identity matrix
I = np.array([[1, 0], [0, 1]])
sigma_x = np.array([[0, 1], [1, 0]])
sigma_y = np.array([[0, -1j], [1j, 0]])
sigma_z = np.array([[1, 0], [0, -1]])

print("Identity Matrix (I):")
print(I)
print("\nPauli Sigma_x (σ_x):")
print(sigma_x)
print("\nPauli Sigma_y (σ_y):")
print(sigma_y)
print("\nPauli Sigma_z (σ_z):")
print(sigma_z)

# Calculate the rotation operator D_x(-pi/2)
# D_x(-pi/2) = (1/sqrt(2)) * (I + i * sigma_x)
rotation_operator_x = (1 / np.sqrt(2)) * (I + 1j * sigma_x)

print("\nCalculated Rotation Operator D_x(-π/2):")
print(rotation_operator_x)

# Verify its properties (e.g., unitarity: D_dagger * D = I)
conjugate_transpose_rot_op = rotation_operator_x.conj().T
identity_check = np.dot(conjugate_transpose_rot_op, rotation_operator_x)

print("\nIs the rotation operator unitary? (D_dagger * D):")
print(identity_check)
print("Close to Identity: ", np.allclose(identity_check, I))


### b. Construct the matrix representation of S_z when the eigenkets of S_y are used as base vectors.

Solution:

**b. Change of Basis for the S_z Operator**
Let $ \{|y,+⟩,|y,-⟩\} $ be the orthonormal basis of $ S_y $ eigenkets. In the standard $ S_z $ basis $ \{|+\rangle,|-\rangle\} $, these are defined as:

$ |y, \pm\rangle = 1/\sqrt{2} (|\pm\rangle \pm i |\mp\rangle) $

The matrix elements of $ S_z $ in the $ S_y $ basis are given by $ M_{ij}=\langle y,i|S_z|y,j\rangle $. Recalling that $ S_z=\hbar/2 (|+\rangle\langle+|-|-\rangle\langle-|) $, Let's use the transformation matrix $ U $ where $ |y,\pm\rangle $ are columns:

$ U=1/\sqrt{2} \begin{pmatrix} 1 & 1 \\ i & -i \end{pmatrix}, U^\dagger=1/\sqrt{2} \begin{pmatrix} 1 & -i \\ 1 & i \end{pmatrix} $

The representation is $ S_z^{(y)} = U^\dagger S_z^{(z)} U $:

$ S_z^{(y)} = 1/2 \begin{pmatrix} 1 & -i \\ 1 & i \end{pmatrix} \begin{pmatrix} \hbar/2 & 0 \\ 0 & -\hbar/2 \end{pmatrix} \begin{pmatrix} 1 & 1 \\ i & -i \end{pmatrix} $

$ S_z^{(y)} = \hbar/4 \begin{pmatrix} 1 & -i \\ 1 & i \end{pmatrix} \begin{pmatrix} 1 & 1 \\ -i & i \end{pmatrix} $

In the $ S_y $ basis, the operator $ S_z $ is represented by the matrix:

$ S_z = \hbar/2 \begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}=\hbar/2 \sigma_x $

This result is consistent with the cyclic permutation of indices in the $ SO(3) $ symmetry of spin space.

In [ ]:
# Define hbar (reduced Planck constant) for the calculation
hbar_val = 1.054571817e-34 # J*s

# Define the S_z operator in the S_z basis (S_z^(z))
S_z_z_basis = (hbar_val / 2) * sigma_z
print("S_z operator in S_z basis:")
print(S_z_z_basis)

# Define the transformation matrix U and its conjugate transpose U_dagger
U = (1 / np.sqrt(2)) * np.array([[1, 1], [1j, -1j]])
U_dagger = U.conj().T

print("\nTransformation Matrix U:")
print(U)
print("\nConjugate Transpose U_dagger:")
print(U_dagger)

# Calculate S_z in the S_y basis (S_z^(y)) using S_z^(y) = U_dagger * S_z^(z) * U
S_z_y_basis = np.dot(U_dagger, np.dot(S_z_z_basis, U))

print("\nS_z operator in S_y basis:")
print(S_z_y_basis)

# Verify the result: S_z^(y) = (hbar/2) * sigma_x
expected_S_z_y_basis = (hbar_val / 2) * sigma_x

print("\nExpected S_z operator in S_y basis ((hbar/2) * sigma_x):")
print(expected_S_z_y_basis)

print("\nIs the calculated S_z_y_basis equal to the expected one?")
print(np.allclose(S_z_y_basis, expected_S_z_y_basis))


### Visualizing State Trajectory on Bloch Sphere for $D_x(-\pi/2)$ Rotation

Building upon the derivation in 1.26a, we will now visualize the effect of the rotation operator $D_x(\phi) = \exp(-i\sigma_x \phi/2)$ on a quantum state. We will start with a simple state, the spin-up state $|+\rangle = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$, and animate its trajectory on the Bloch sphere as the rotation angle $\phi$ varies from $0$ to $-\pi/2$. This will show how the state vector rotates around the x-axis.

The Bloch vector for a general state $c_1|+\rangle + c_2|-\rangle$ is given by $(x,y,z)$ where:
$x = 2\text{Re}(c_1^* c_2)$
$y = 2\text{Im}(c_1^* c_2)$
$z = |c_1|^2 - |c_2|^2$

Applying the rotation operator $D_x(\phi)$ to the spin-up state $|+\rangle$ gives:
$D_x(\phi)|+\rangle = \begin{pmatrix} \cos(\phi/2) \\ -i\sin(\phi/2) \end{pmatrix}$

From this, we can find the coordinates of the Bloch vector as a function of $\phi$:
$x = 0$
$y = -\sin(\phi)$
$z = \cos(\phi)$

This confirms that the state vector traces a path in the y-z plane, starting from the north pole $(0,0,1)$ at $\phi=0$ and moving towards $(0,1,0)$ (the positive y-axis) at $\phi=-\pi/2$. This movement corresponds to a rotation about the x-axis.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Define Pauli matrices and Identity matrix (re-defining for self-containment)
I = np.array([[1, 0], [0, 1]], dtype=complex)
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)

# Function to draw a Bloch sphere base
def plot_bloch_sphere_base(ax, title="Bloch Sphere with Rotating Spin State"):
    # Draw the sphere
    u = np.linspace(0, 2 * np.pi, 50)
    v = np.linspace(0, np.pi, 50)
    x_sphere = np.outer(np.cos(u), np.sin(v))
    y_sphere = np.outer(np.sin(u), np.sin(v))
    z_sphere = np.outer(np.ones(np.size(u)), np.cos(v))
    ax.plot_surface(x_sphere, y_sphere, z_sphere, color='lightgray', alpha=0.1, rstride=5, cstride=5)

    # Draw axes
    ax.plot([-1.1, 1.1], [0, 0], [0, 0], 'r--', linewidth=2) # X-axis (Rotation axis highlighted in red)
    ax.plot([0, 0], [-1.1, 1.1], [0, 0], 'k--') # Y-axis
    ax.plot([0, 0], [0, 0], [-1.1, 1.1], 'k--') # Z-axis

    # Labels for axes
    ax.text(0, 0, 1.2, r'$|\uparrow\rangle$', color='blue') # Spin up
    ax.text(0, 0, -1.2, r'$|\downarrow\rangle$', color='red') # Spin down
    ax.text(1.3, 0, 0, r'$X$', color='red', fontsize=12) # X-axis label
    ax.text(0, 1.3, 0, r'$Y$', color='black', fontsize=12) # Y-axis label
    ax.text(0, 0, 1.3, r'$Z$', color='blue', fontsize=12) # Z-axis label

    ax.set_box_aspect([1,1,1]) # Equal aspect ratio
    ax.set_axis_off()
    ax.set_title(title)

# --- Animation Setup ---
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection='3d')

plot_bloch_sphere_base(ax, title=r'Rotation of $|\uparrow\rangle$ by $D_x(\phi)$ on Bloch Sphere')

# Initial state: spin-up ($|+\rangle$)
initial_state_vector = np.array([[1], [0]], dtype=complex)

# Initialize the spin state vector on the Bloch sphere
quiver = ax.quiver(0, 0, 0, 0, 0, 0, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3, label='State Vector')

# Initialize text for displaying the rotation angle
angle_text = ax.text2D(0.05, 0.95, '', transform=ax.transAxes, color='darkred', fontsize=12)

# Initialize line for the trajectory path
trajectory_line, = ax.plot([], [], [], 'cyan', linestyle='-', linewidth=2, alpha=0.8, label='Trajectory')

# Lists to store trajectory points
trajectory_x, trajectory_y, trajectory_z = [], [], []

# Markers for initial and final states
start_marker, = ax.plot([], [], [], 'o', color='blue', markersize=8, label=r'Initial State $|\uparrow\rangle$')
end_marker, = ax.plot([], [], [], 's', color='purple', markersize=8, label=r'Final State after $-\pi/2$ Rotation')

ax.legend()

# --- Animation Functions ---
def init_rotation_anim():
    global quiver, trajectory_x, trajectory_y, trajectory_z
    # Clear previous quiver if exists
    if 'quiver' in globals() and quiver is not None:
        quiver.remove()

    # Initial vector (for phi=0, state is (0,0,1))
    # The Bloch vector for spin-up is (0,0,1)
    quiver = ax.quiver(0, 0, 0, 0, 0, 1, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3)
    angle_text.set_text(r'Rotation Angle $\phi=0$')

    # Clear trajectory data
    trajectory_x, trajectory_y, trajectory_z = [], [], []
    trajectory_line.set_data([], [])

    # Set initial state marker
    start_marker.set_data([0], [0])
    start_marker.set_3d_properties([1]) # Start at (0,0,1)

    # Calculate the final state (phi = -pi/2) for the end marker
    final_phi = -np.pi/2
    rotation_operator_final = np.cos(final_phi/2) * I - 1j * np.sin(final_phi/2) * sigma_x
    rotated_state_final = np.dot(rotation_operator_final, initial_state_vector)
    c1_final, c2_final = rotated_state_final[0, 0], rotated_state_final[1, 0]
    x_final = 2 * np.real(np.conj(c1_final) * c2_final)
    y_final = 2 * np.imag(np.conj(c1_final) * c2_final)
    z_final = np.real(np.conj(c1_final) * c1_final - np.conj(c2_final) * c2_final)
    end_marker.set_data([x_final], [y_final])
    end_marker.set_3d_properties([z_final]) # End at (0,1,0)

    return quiver, angle_text, trajectory_line, start_marker, end_marker

def update_rotation_anim(frame):
    global quiver, trajectory_x, trajectory_y, trajectory_z
    # Rotation angle varies from 0 to -pi/2
    phi = -np.pi/2 * (frame / num_frames_rotation_anim)

    # Calculate the rotated state |ψ(phi)〉
    # D_x(phi) = cos(phi/2)*I - i*sin(phi/2)*sigma_x
    rotation_operator = np.cos(phi/2) * I - 1j * np.sin(phi/2) * sigma_x
    rotated_state = np.dot(rotation_operator, initial_state_vector)

    # Extract components c1 and c2
    c1 = rotated_state[0, 0]
    c2 = rotated_state[1, 0]

    # Calculate Bloch sphere coordinates (x, y, z)
    # These simplify to (0, -sin(phi), cos(phi)) for initial_state = |up>
    x_vec = 2 * np.real(np.conj(c1) * c2)
    y_vec = 2 * np.imag(np.conj(c1) * c2)
    z_vec = np.real(np.conj(c1) * c1 - np.conj(c2) * c2)

    # Remove old quiver and draw new one
    if quiver is not None:
        quiver.remove()
    quiver = ax.quiver(0, 0, 0, x_vec, y_vec, z_vec, color='darkgreen', length=1, arrow_length_ratio=0.1, linewidth=3)

    # Update angle text
    angle_text.set_text(rf'Rotation Angle $\phi={phi/np.pi:.2f}\pi$')

    # Store current point for trajectory
    trajectory_x.append(x_vec)
    trajectory_y.append(y_vec)
    trajectory_z.append(z_vec)
    trajectory_line.set_data(trajectory_x, trajectory_y)
    trajectory_line.set_3d_properties(trajectory_z)

    # Rotate view for better perspective during animation
    ax.view_init(elev=30, azim=-90 - np.degrees(phi) * 2)

    return quiver, angle_text, trajectory_line, start_marker, end_marker

# Create the animation
num_frames_rotation_anim = 100 # Number of frames
ani_rotation = FuncAnimation(fig, update_rotation_anim, frames=num_frames_rotation_anim,
                               init_func=init_rotation_anim, blit=False, interval=100)

# Display the animation in the notebook
plt.close(fig) # Prevent static plot from showing
html_ani_rotation = HTML(ani_rotation.to_jshtml())
display(html_ani_rotation)

# Save the animation as a GIF
print("Saving Bloch sphere rotation animation as GIF...")
ani_rotation.save('bloch_sphere_rotation_dx.gif', writer='pillow', fps=10)
print("GIF saved: bloch_sphere_rotation_dx.gif")

## Infinitesimal Boost Operator and Momentum Space Transformations (Problem 1.34)

In quantum mechanics, infinitesimal boost operators are crucial for understanding transformations in momentum space. Problem 1.34 defines such an operator as $B(\text{d}\mathbf{p}') = \mathbf{I} + i\mathbf{W} \cdot \text{d}\mathbf{p}'$, where $\mathbf{W}$ is a Hermitian operator. This operator acts to change a momentum eigenstate $|\mathbf{p}'\rangle$ to $|\mathbf{p}' + \text{d}\mathbf{p}'\rangle$. We will verify its fundamental properties: unitarity and the inverse property.

For this simulation, we'll represent $\mathbf{W}$ by a generic Hermitian matrix and $\text{d}\mathbf{p}'$ as a small scalar increment, demonstrating these properties numerically for a simplified case.

In [ ]:
import numpy as np

# Define a small scalar to represent the infinitesimal dp'
dp_scalar = 1e-9 # A very small number representing an infinitesimal change

# We need a Hermitian matrix for W. Let's use a simple 2x2 Hermitian matrix.
# A Pauli matrix (e.g., sigma_x) is a good example of a Hermitian operator.
# For a more general Hermitian matrix: A = A_dagger.
# Example: [[a, b+ci], [b-ci, d]] where a, d are real.
W_matrix = np.array([[2, 1 - 0.5j],
                     [1 + 0.5j, 3]], dtype=complex)

# Identity matrix matching the dimension of W_matrix
I_matrix = np.identity(W_matrix.shape[0], dtype=complex)

# Construct the infinitesimal boost operator B(dp') = I + i * W * dp'
# We treat dp' as a scalar here for demonstration purposes.
B_matrix = I_matrix + 1j * W_matrix * dp_scalar

print("Infinitesimal dp':", dp_scalar)
print("\nGeneric Hermitian W_matrix:")
print(W_matrix)
print("\nConstructed Boost Operator B_matrix:")
print(B_matrix)

### Verification of Unitarity

For an operator to be unitary, $B^\dagger B = \mathbf{I}$, where $B^\dagger$ is the conjugate transpose of $B$. For an infinitesimal operator $B = \mathbf{I} + i\mathbf{W} \text{d}p'$, its conjugate transpose is $B^\dagger = \mathbf{I} - i\mathbf{W}^\dagger \text{d}p'$. Since $\mathbf{W}$ is Hermitian, $\mathbf{W}^\dagger = \mathbf{W}$.

Then $B^\dagger B = (\mathbf{I} - i\mathbf{W} \text{d}p') (\mathbf{I} + i\mathbf{W} \text{d}p') = \mathbf{I}^2 + i\mathbf{W} \text{d}p' - i\mathbf{W} \text{d}p' + (i\mathbf{W} \text{d}p')(-i\mathbf{W} \text{d}p') = \mathbf{I} + \mathbf{W}^2 (\text{d}p')^2$.

For infinitesimal $\text{d}p'$, the $(\text{d}p')^2$ term is negligible (approaches zero much faster than $\text{d}p'$). Thus, $B^\dagger B \approx \mathbf{I}$.

In [ ]:
# Calculate the conjugate transpose of B_matrix
B_dagger_matrix = B_matrix.conj().T

# Calculate B_dagger * B
product_unitarity = np.dot(B_dagger_matrix, B_matrix)

print("B_dagger_matrix:")
print(B_dagger_matrix)
print("\nProduct B_dagger * B:")
print(product_unitarity)

# Check if it's close to the Identity matrix, neglecting terms of order (dp_scalar)^2
# np.allclose considers small floating point differences
print("\nIs B_dagger * B approximately equal to Identity?")
print(np.allclose(product_unitarity, I_matrix, atol=dp_scalar**1.5)) # Allow for (dp_scalar)^2 differences

# Explicitly show the deviation from identity
deviation_unitarity = product_unitarity - I_matrix
print("\nDeviation from Identity (should be order dp_scalar^2):")
print(deviation_unitarity)

### Verification of Inverse Property

The inverse of an infinitesimal boost operator $B(\text{d}\mathbf{p}')$ should correspond to a boost in the opposite direction, i.e., $B^{-1}(\text{d}\mathbf{p}') = B(-\text{d}\mathbf{p}') = \mathbf{I} - i\mathbf{W} \text{d}p'$.

Multiplying $B^{-1}$ by $B$: $B^{-1} B = (\mathbf{I} - i\mathbf{W} \text{d}p') (\mathbf{I} + i\mathbf{W} \text{d}p') = \mathbf{I} + (i\mathbf{W}\text{d}p') - (i\mathbf{W}\text{d}p') - (i\mathbf{W}\text{d}p')(i\mathbf{W}\text{d}p') = \mathbf{I} + \mathbf{W}^2 (\text{d}p')^2$. Again, for infinitesimal $\text{d}p'$, terms of order $(\text{d}p')^2$ are negligible, so $B^{-1} B \approx \mathbf{I}$.

In [ ]:
# Define the inverse boost operator B_inverse = I - i * W * dp'
B_inverse_matrix = I_matrix - 1j * W_matrix * dp_scalar

# Calculate B_inverse * B
product_inverse = np.dot(B_inverse_matrix, B_matrix)

print("B_inverse_matrix:")
print(B_inverse_matrix)
print("\nProduct B_inverse * B:")
print(product_inverse)

# Check if it's close to the Identity matrix
print("\nIs B_inverse * B approximately equal to Identity?")
print(np.allclose(product_inverse, I_matrix, atol=dp_scalar**1.5))

# Explicitly show the deviation from identity
deviation_inverse = product_inverse - I_matrix
print("\nDeviation from Identity (should be order dp_scalar^2):")
print(deviation_inverse)

### Dimensional Analysis and Identification of $\mathbf{W}$

The problem states that through dimensional analysis, the generator $\mathbf{W}$ can be identified with the position operator $\mathbf{x}$ divided by the reduced Planck constant $\hbar$, i.e., $\mathbf{W} = \mathbf{x}/\hbar$.

In this context, $\mathbf{x}$ and $\mathbf{p}$ are fundamental non-commuting operators related by canonical commutation relations. While we have simulated the algebraic properties of the general boost operator, the full simulation of the canonical commutation relations and their implications requires a more advanced symbolic or functional approach (e.g., using `sympy` to handle non-commutative operators or defining operators' action on a Hilbert space), which is beyond the scope of a direct numerical `numpy` demonstration with simple matrices.

## Simulating the Action of Infinitesimal Operators

### Infinitesimal Translation Operator $j(	ext{d}\mathbf{x}')$ (Problem 1.33)

The infinitesimal translation operator $j(	ext{d}\mathbf{x}') = 	ext{exp}(-i \mathbf{p} \cdot \text{d}\mathbf{x}' / \hbar)$ effectively shifts a wave function in position space. If a state $|\alpha\rangle$ is translated to $j(	ext{d}\mathbf{x}')|\alpha\rangle$, then its position-space wave function $\psi(\mathbf{x})$ transforms as $\psi(\mathbf{x}) \rightarrow \psi(\mathbf{x} - \text{d}\mathbf{x}')$.

As derived in Problem 1.33, this leads to the following transformations for expectation values:

*   $\langle\mathbf{x}\rangle \rightarrow \langle\mathbf{x}\rangle + \text{d}\mathbf{x}'$
*   $\langle\mathbf{p}\rangle \rightarrow \langle\mathbf{p}\rangle$ (momentum expectation value remains unchanged)

We will visualize this by taking a simple one-dimensional Gaussian wave packet, shifting it by a small amount, and observing the change in its position and momentum expectation values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Using numpy.trapezoid for integration

# --- Constants ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in position space ---
def gaussian_wavepacket_x(x, x0, sigma):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - x0)**2 / (2 * sigma**2))

# --- Simulation parameters ---
x_values = np.linspace(-50, 50, 50000) # Position range - Increased range for accuracy
x0_initial = 0.0 # Initial center of the wave packet
sigma_x = 1.0 # Width of the wave packet
dx_val = 2.0 # Infinitesimal translation amount

# --- Original wave packet --- (psi_original is already the PDF)
psi_original = gaussian_wavepacket_x(x_values, x0_initial, sigma_x)
probability_density_original = psi_original # Correct: use the PDF directly

# --- Calculate expectation values for original state ---
x_exp_original = np.trapezoid(x_values * probability_density_original, x=x_values)
# For a stationary Gaussian, <p> is 0. Numerical derivative for <p> is complex;
# we rely on the analytical result from the problem statement for <p> invariance.
p_exp_original = 0.0 # Analytically for a stationary Gaussian

print("--- Original State ---")
print(f"Initial <x>: {x_exp_original:.2f}")
print(f"Initial <p>: {p_exp_original:.2f} (analytically for stationary Gaussian)")

# --- Translated wave packet --- (psi_translated is already the PDF)
# The operator j(dx') translates psi(x) to psi(x - dx')
psi_translated = gaussian_wavepacket_x(x_values, x0_initial + dx_val, sigma_x)
probability_density_translated = psi_translated # Correct: use the PDF directly

# --- Calculate expectation values for translated state ---
x_exp_translated = np.trapezoid(x_values * probability_density_translated, x=x_values)
# <p> is expected to remain unchanged under spatial translation
p_exp_translated = 0.0 # Analytically

print("\n--- Translated State ---")
print(f"Translated <x>: {x_exp_translated:.2f} (expected: {x_exp_original + dx_val:.2f})")
print(f"Translated <p>: {p_exp_translated:.2f} (expected: {p_exp_original:.2f})")

# --- Plotting the wave packets ---
plt.figure(figsize=(10, 6))
plt.plot(x_values, probability_density_original, label=r'Original $|\psi(x)|^2$', color='blue')
plt.plot(x_values, probability_density_translated, label=fr'Translated $|\psi(x-\text{{d}}x)|^2$ (dx={dx_val:.1f})', linestyle='--', color='red')
plt.axvline(x_exp_original, color='blue', linestyle=':', label=fr'Original $<x>$ = {x_exp_original:.2f}')
plt.axvline(x_exp_translated, color='red', linestyle=':', label=fr'Translated $<x>$ = {x_exp_translated:.2f}')

plt.title(r'Effect of Infinitesimal Translation Operator $j(\text{d}x)$ on Wave Packet')
plt.xlabel('Position $x$')
plt.ylabel(r'Probability Density $|\psi(x)|^2$')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Constants (re-using hbar_val from previous cells) ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in momentum space ---
def gaussian_wavepacket_p(p, p0, sigma_p):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma_p * np.sqrt(2 * np.pi))) * np.exp(-(p - p0)**2 / (2 * sigma_p**2))

# --- Simulation parameters ---
p_values = np.linspace(-50 * hbar_val, 50 * hbar_val, 50000) # Momentum range
p0_initial = 1.0 * hbar_val # Initial center of the wave packet in momentum
sigma_p = 0.5 * hbar_val # Width of the wave packet in momentum
dx_val = 2.0 # Infinitesimal spatial translation amount (arbitrary, for demonstration)

# --- Original wave packet in momentum space ---
phi_original = gaussian_wavepacket_p(p_values, p0_initial, sigma_p)
probability_density_p_original = phi_original # Already PDF

# --- Apply translation operator in momentum space ---
# j(dx') acts as exp(-i p * dx' / hbar) in momentum space
phase_factor = np.exp(-1j * p_values * dx_val / hbar_val)
phi_translated_complex = phase_factor * phi_original # Multiply by phase factor

# The probability density should be unchanged!
probability_density_p_translated = np.abs(phi_translated_complex)**2

# --- Plotting the wave packets in momentum space ---
plt.figure(figsize=(10, 6))
plt.plot(p_values, probability_density_p_original, label=r'Original $|φ(p)|^2$', color='blue')
plt.plot(p_values, probability_density_p_translated, label=fr'Translated $|φ(p)|^2$ with phase (dx={dx_val:.1f})', linestyle='--', color='red')

plt.title(r'Effect of Infinitesimal Translation Operator $j(	ext{d}x)$ on Momentum-Space Wave Packet')
plt.xlabel('Momentum $p$')
plt.ylabel(r'Probability Density $|φ(p)|^2$')
plt.legend()
plt.grid(True)
plt.show()

### Infinitesimal Translation Operator $j(dx)$ in Momentum Space

Cuando el operador de traslación infinitesimal $j(\text{d}x)$ actúa sobre un estado cuántico, su representación en el **espacio de momentos**, $\phi(p)$, se transforma adquiriendo un factor de fase: $\phi(p) \rightarrow e^{-i p \cdot \text{d}x / \hbar} \phi(p)$.

La observación crucial aquí es que la **densidad de probabilidad** en el espacio de momentos, $|\phi(p)|^2$, permanece **sin cambios** por esta operación. Esto se debe a que $|e^{-i p \cdot \text{d}x / \hbar} \phi(p)|^2 = |e^{-i p \cdot \text{d}x / \hbar}|^2 |\phi(p)|^2 = (1) |\phi(p)|^2$.

Este resultado es consistente con la predicción teórica de que, si bien la traslación espacial desplaza el valor esperado de la posición $\langle x \rangle$, deja el valor esperado del momento $\langle p \rangle$ invariante. Dado que la distribución de probabilidad del momento en sí misma no cambia, el valor esperado derivado de ella permanece igual.

### Infinitesimal Boost Operator $B(	ext{d}\mathbf{p}')$ (Problem 1.34)

### Animation: Infinitesimal Translation Operator $\mathcal{T}(dx)$ on Position-Space Wave Packet

This animation visualizes the effect of the infinitesimal translation operator by showing a 1D Gaussian wave packet shifting its position in space. As the operator $\mathcal{T}(dx)$ acts, the center of the wave packet moves by $dx$. Concurrently, the expectation value of momentum $\langle p \rangle$ for this wave packet remains unchanged, reflecting that a pure spatial translation does not alter the momentum characteristics.

We will animate the probability density $|\psi(x)|^2$ in position space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Constants (re-defined for self-containment) ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in position space (re-defined for self-containment) ---
def gaussian_wavepacket_x(x, x0, sigma):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - x0)**2 / (2 * sigma**2))

# --- Simulation parameters from original cell (re-defined for self-containment) ---
x_values = np.linspace(-50, 50, 50000) # Position range - Increased range for accuracy
x0_initial = 0.0 # Initial center of the wave packet
sigma_x = 1.0 # Width of the wave packet
dx_val = 2.0 # Infinitesimal translation amount

# --- Simulation parameters for animation ---
x0_start_anim = x0_initial
x0_end_anim = x0_initial + dx_val
num_frames_anim = 100

# --- Animation Setup ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(x_values.min(), x_values.max())
ax.set_ylim(0, gaussian_wavepacket_x(x0_initial, x0_initial, sigma_x).max() * 1.1)
ax.set_xlabel('Position $x$')
ax.set_ylabel(r'Probability Density $|\psi(x)|^2$')
ax.set_title(r'Animation of Infinitesimal Translation $\mathcal{T}(dx)$')
ax.grid(True)

line, = ax.plot([], [], 'r-', lw=2, label=r'$|\psi(x, t)|^2$')
center_line = ax.axvline(x=x0_start_anim, color='blue', linestyle=':', label=r'Center $<x>$') # Updated init
angle_text = ax.text(0.05, 0.95, '', transform=ax.transAxes, color='darkred', fontsize=12)

# --- Animation Functions ---
def init_translation_anim():
    line.set_data([], [])
    center_line.set_xdata([x0_start_anim]) # Fix: Pass a list
    angle_text.set_text('')
    return line, center_line, angle_text

def update_translation_anim(frame):
    current_x0 = x0_start_anim + (x0_end_anim - x0_start_anim) * (frame / (num_frames_anim - 1))

    # Calculate the probability density for the current x0
    current_psi_sq = gaussian_wavepacket_x(x_values, current_x0, sigma_x)

    line.set_data(x_values, current_psi_sq)
    center_line.set_xdata([current_x0]) # Fix: Pass a list
    angle_text.set_text(fr'$<x> = {current_x0:.2f}$')

    return line, center_line, angle_text

# Create the animation
ani_translation = FuncAnimation(fig, update_translation_anim, frames=num_frames_anim,
                               init_func=init_translation_anim, blit=True, interval=50)

# Display the animation in the notebook
plt.close(fig) # Prevent static plot from showing
html_ani_translation = HTML(ani_translation.to_jshtml())
display(html_ani_translation)

print("Saving translation animation as GIF...")
ani_translation.save('translation_wavepacket_animation.gif', writer='pillow', fps=20)
print("GIF saved: translation_wavepacket_animation.gif")

### Animation: Infinitesimal Boost Operator $\mathcal{B}(dp)$ on Momentum-Space Wave Packet

This animation illustrates the effect of the infinitesimal boost operator by showing a 1D Gaussian wave packet shifting its momentum in momentum space. As the operator $\mathcal{B}(dp)$ acts, the center of the wave packet moves by $dp$. Importantly, the expectation value of position $\langle x \rangle$ for this wave packet remains unchanged, although its phase in position space would show a linearly increasing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Constants (re-defined for self-containment) ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in momentum space (re-defined for self-containment) ---
def gaussian_wavepacket_p(p, p0, sigma_p):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma_p * np.sqrt(2 * np.pi))) * np.exp(-(p - p0)**2 / (2 * sigma_p**2))

# --- Simulation parameters from original cell (re-defined for self-containment) ---
p_values = np.linspace(-50 * hbar_val, 50 * hbar_val, 50000) # Momentum range - Increased range for accuracy
p0_initial = 1.0 * hbar_val # Initial center of the wave packet in momentum
sigma_p = 0.5 * hbar_val # Width of the wave packet in momentum
dp_val = 2.0 * hbar_val # Infinitesimal boost amount

# --- Simulation parameters for animation ---
p0_start_anim = p0_initial
p0_end_anim = p0_initial + dp_val
num_frames_anim = 100

# --- Animation Setup ---
fig, ax = plt.subplots(figsize=(10, 6))
ax.set_xlim(p_values.min(), p_values.max())
ax.set_ylim(0, gaussian_wavepacket_p(p0_initial, p0_initial, sigma_p).max() * 1.1)
ax.set_xlabel('Momentum $p$')
ax.set_ylabel(r'Probability Density $|\phi(p)|^2$')
ax.set_title(r'Animation of Infinitesimal Boost $\mathcal{B}(dp)$')
ax.grid(True)

line, = ax.plot([], [], 'g-', lw=2, label=r'$|\phi(p, t)|^2$')
center_line = ax.axvline(p0_start_anim, color='purple', linestyle=':', label=r'Center $<p>$')
angle_text = ax.text(0.05, 0.95, '', transform=ax.transAxes, color='darkred', fontsize=12)

# --- Animation Functions ---
def init_boost_anim():
    line.set_data([], [])
    center_line.set_xdata([p0_start_anim]) # Fix: Pass a list
    angle_text.set_text('')
    return line, center_line, angle_text

def update_boost_anim(frame):
    current_p0 = p0_start_anim + (p0_end_anim - p0_start_anim) * (frame / (num_frames_anim - 1))

    # Calculate the probability density for the current p0
    current_phi_sq = gaussian_wavepacket_p(p_values, current_p0, sigma_p)

    line.set_data(p_values, current_phi_sq)
    center_line.set_xdata([current_p0]) # Fix: Pass a list
    angle_text.set_text(fr'$<p> = {current_p0:.2e}$')

    return line, center_line, angle_text

# Create the animation
ani_boost = FuncAnimation(fig, update_boost_anim, frames=num_frames_anim,
                           init_func=init_boost_anim, blit=True, interval=50)

# Display the animation in the notebook
plt.close(fig) # Prevent static plot from showing
html_ani_boost = HTML(ani_boost.to_jshtml())
display(html_ani_boost)

print("Saving boost animation as GIF...")
ani_boost.save('boost_wavepacket_animation.gif', writer='pillow', fps=20)
print("GIF saved: boost_wavepacket_animation.gif")

The infinitesimal boost operator $B(	ext{d}\mathbf{p}') = \mathbf{I} + i/\hbar \mathbf{x} \cdot \text{d}\mathbf{p}'$ (as defined in Problem 1.34) acts to shift a state in momentum space. If a state $|\alpha\rangle$ is boosted to $B(	ext{d}\mathbf{p}')|\alpha\rangle$, then its momentum-space wave function $\phi(\mathbf{p})$ transforms as $\phi(\mathbf{p}) \rightarrow \phi(\mathbf{p} - \text{d}\mathbf{p}')$.

As derived in the thought process, this leads to the following transformations for expectation values:

*   $\langle\mathbf{p}\rangle \rightarrow \langle\mathbf{p}\rangle + \text{d}\mathbf{p}'$
*   $\langle\mathbf{x}\rangle \rightarrow \langle\mathbf{x}\rangle$ (position expectation value remains unchanged)

We will visualize this by taking a simple one-dimensional Gaussian wave packet in momentum space, shifting it by a small amount, and observing the change in its momentum and position expectation values.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Using numpy.trapezoid for integration

from scipy.fft import fft, ifft, fftfreq

# --- Constants ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in momentum space ---
def gaussian_wavepacket_p(p, p0, sigma_p):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma_p * np.sqrt(2 * np.pi))) * np.exp(-(p - p0)**2 / (2 * sigma_p**2))

# --- Simulation parameters ---
p_values = np.linspace(-50 * hbar_val, 50 * hbar_val, 50000) # Momentum range - Increased range for accuracy
p0_initial = 1.0 * hbar_val # Initial center of the wave packet in momentum
sigma_p = 0.5 * hbar_val # Width of the wave packet in momentum
dp_val = 2.0 * hbar_val # Infinitesimal boost amount

# --- Original wave packet in momentum space --- (phi_original is already the PDF)
phi_original = gaussian_wavepacket_p(p_values, p0_initial, sigma_p)
probability_density_p_original = phi_original # Correct: use the PDF directly

# --- Calculate expectation values for original state ---
p_exp_original = np.trapezoid(p_values * probability_density_p_original, x=p_values)
# For <x> in a momentum-space Gaussian centered at p0, if p0 != 0, it represents a traveling wave.
# The expectation value <x> is 0 if the phase is set such that the center of the wavepacket is at x=0
# or if one computes <x> from the momentum-space wavefunction using <x> = i*hbar*int(phi*(dp/dp)phi)dp
# For simplicity, we rely on the analytical result from the problem statement for <x> invariance.
x_exp_original = 0.0 # Analytically for this simplified case at x=0

print("--- Original State ---")
print(f"Initial <p>: {p_exp_original:.2e} (expected: {p0_initial:.2e})") # Corrected expected value for clarity
print(f"Initial <x>: {x_exp_original:.2f} (analytically for center at x=0)")

# --- Boosted wave packet in momentum space --- (phi_boosted is already the PDF)
# The operator B(dp') boosts phi(p) to phi(p - dp')
phi_boosted = gaussian_wavepacket_p(p_values, p0_initial + dp_val, sigma_p)
probability_density_p_boosted = phi_boosted # Correct: use the PDF directly

# --- Calculate expectation values for boosted state ---
p_exp_boosted = np.trapezoid(p_values * probability_density_p_boosted, x=p_values)
# <x> is expected to remain unchanged under momentum boost
x_exp_boosted = 0.0 # Analytically

print("\n--- Boosted State ---")
print(f"Boosted <p>: {p_exp_boosted:.2e} (expected: {p0_initial + dp_val:.2e})")
print(f"Boosted <x>: {x_exp_boosted:.2f} (expected: {x_exp_original:.2f})")

# --- Plotting the wave packets in momentum space ---
plt.figure(figsize=(10, 6))
plt.plot(p_values, probability_density_p_original, label=r'Original $|\phi(p)|^2$', color='blue')
plt.plot(p_values, probability_density_p_boosted, label=fr'Boosted $|\phi(p-\text{{d}}p)|^2$ (dp={dp_val:.2e})', linestyle='--', color='red')
plt.axvline(p_exp_original, color='blue', linestyle=':', label=fr'Original $<p>$ = {p_exp_original:.2e}')
plt.axvline(p_exp_boosted, color='red', linestyle=':', label=fr'Boosted $<p>$ = {p_exp_boosted:.2e}')

plt.title(r'Effect of Infinitesimal Boost Operator $B(\text{d}p)$ on Wave Packet')
plt.xlabel('Momentum $p$')
plt.ylabel(r'Probability Density $|\phi(p)|^2$')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --- Constants (re-using hbar_val from previous cells) ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in position space ---
def gaussian_wavepacket_x(x, x0, sigma):
    # This function returns a normalized probability density function (PDF)
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - x0)**2 / (2 * sigma**2))

# --- Simulation parameters ---
x_values = np.linspace(-50, 50, 50000) # Position range
x0_initial = 0.0 # Initial center of the wave packet
sigma_x = 1.0 # Width of the wave packet
dp_val = 2.0 * hbar_val # Infinitesimal momentum boost amount (arbitrary, for demonstration)

# --- Original wave packet in position space ---
psi_original = gaussian_wavepacket_x(x_values, x0_initial, sigma_x)
probability_density_x_original = psi_original # Already PDF

# --- Apply boost operator in position space ---
# B(dp') acts as exp(i x * dp' / hbar) in position space
phase_factor = np.exp(1j * x_values * dp_val / hbar_val)
psi_boosted_complex = phase_factor * psi_original # Multiply by phase factor

# The probability density should be unchanged!
probability_density_x_boosted = np.abs(psi_boosted_complex)**2

# --- Plotting the wave packets in position space ---
plt.figure(figsize=(10, 6))
plt.plot(x_values, probability_density_x_original, label=r'Original $|ψ(x)|^2$', color='blue')
plt.plot(x_values, probability_density_x_boosted, label=fr'Boosted $|ψ(x)|^2$ with phase (dp={dp_val:.2e})', linestyle='--', color='red')

plt.title(r'Effect of Infinitesimal Boost Operator $B(	ext{d}p)$ on Position-Space Wave Packet')
plt.xlabel('Position $x$')
plt.ylabel(r'Probability Density $|ψ(x)|^2$')
plt.legend()
plt.grid(True)
plt.show()

### Infinitesimal Boost Operator $B(dp)$ in Position Space

Cuando el operador de impulso infinitesimal $B(\text{d}p)$ actúa sobre un estado cuántico, su representación en el **espacio de posiciones**, $\psi(x)$, se transforma adquiriendo un factor de fase: $\psi(x) \rightarrow e^{i x \cdot \text{d}p / \hbar} \psi(x)$.

Similar al operador de traslación en el espacio de momentos, la **densidad de probabilidad** en el espacio de posiciones, $|\psi(x)|^2$, permanece **sin cambios** por esta operación. Esto se debe a que $|e^{i x \cdot \text{d}p / \hbar} \psi(x)|^2 = |e^{i x \cdot \text{d}p / \hbar}|^2 |\psi(x)|^2 = (1) |\psi(x)|^2$.

Esto es consistente con la predicción teórica de que, si bien un impulso en el momento desplaza el valor esperado del momento $\langle p \rangle$, deja el valor esperado de la posición $\langle x \rangle$ invariante. La distribución de probabilidad de la posición en sí misma no cambia, por lo que su valor esperado permanece igual.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, fftshift, fftfreq

# --- Constants ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in position space ---
def gaussian_wavepacket_x(x, x0, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - x0)**2 / (2 * sigma**2))

# --- Simulation parameters for position space ---
x_values = np.linspace(-10, 10, 1000) # Position range
x0_initial = 0.0 # Center of the wave packet
sigma_x = 0.5 # Width of the wave packet (narrow in x-space)

# --- Generate position space wave packet (probability amplitude) ---
# For a real Gaussian, the probability amplitude psi(x) is also Gaussian.
# We will use sqrt of the PDF for the amplitude, or simply define it as a complex Gaussian if we wanted phase.
# For visualization, we often plot the probability density |psi(x)|^2.
psi_x = np.sqrt(gaussian_wavepacket_x(x_values, x0_initial, sigma_x))

# --- Perform Fourier Transform to get momentum space wave packet ---
# The numerical Fourier Transform requires careful handling of sampling and scaling.
# Ensure dx is consistent with x_values
dx = x_values[1] - x_values[0]

# Calculate the FFT of psi_x
# np.fft.fft returns frequencies in the order [0, 1, ..., N/2-1, -N/2, ..., -1]
# We use fftshift to put zero frequency in the center, and scale for momentum values.
# The normalization factor for the Fourier Transform is 1/sqrt(2*pi*hbar)

# First, calculate the unshifted frequencies and then scale to momentum
freq = fftfreq(len(x_values), d=dx)
# Momentum values are p = hbar * k, where k = 2*pi * (shifted)freq
p_values_raw = 2 * np.pi * hbar_val * freq

# Perform FFT and shift
# The numerical FT of a normalized Gaussian in position space is a normalized Gaussian in momentum space
phi_p_unscaled = fft(psi_x)
phi_p = fftshift(phi_p_unscaled) * dx # Multiply by dx to account for numerical integration and approximate continuous FT
p_values = fftshift(p_values_raw)

# --- Normalization of momentum space wave packet (for probability density) ---
# The probability density in momentum space is |phi(p)|^2
probability_density_p = np.abs(phi_p)**2

# We should ensure total probability is 1 in both spaces
# np.trapezoid(np.abs(psi_x)**2, x_values) should be close to 1
# np.trapezoid(probability_density_p, p_values) should be close to 1

# Renormalize momentum space probability density for proper comparison if necessary
# This makes total probability 1, adjusting for any numerical inaccuracies.
probability_density_p = probability_density_p / np.trapezoid(probability_density_p, p_values)

# --- Plotting the wave packets in both spaces ---
plt.figure(figsize=(12, 5))

# Position Space Plot
plt.subplot(1, 2, 1)
plt.plot(x_values, np.abs(psi_x)**2, color='blue', label=r'$|\psi(x)|^2$')
plt.title('Wave Packet in Position Space')
plt.xlabel('Position $x$')
plt.ylabel('Probability Density')
plt.grid(True)
plt.legend()

# Momentum Space Plot
plt.subplot(1, 2, 2)
plt.plot(p_values, probability_density_p, color='red', label=r'$|\phi(p)|^2$')
plt.title('Wave Packet in Momentum Space (Fourier Transform)')
plt.xlabel('Momentum $p$')
plt.ylabel('Probability Density')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- Constants ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in position space (returns probability density) ---
def gaussian_wavepacket_x_pdf(x, x0, sigma):
    return (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-(x - x0)**2 / (2 * sigma**2))

# --- Simulation parameters ---
x_values = np.linspace(-10, 10, 2000) # Position range, increased points for smoother phase plot
x0_initial = 0.0 # Initial center of the wave packet
sigma_x = 1.0 # Width of the wave packet

# Animation for the net momentum p_net (dp in the problem description)
p_net_start_anim = 0.0 * hbar_val # Start with zero net momentum
p_net_end_anim = 5.0 * hbar_val # End with a significant net momentum for visualization
num_frames_anim = 200 # More frames for smoother phase evolution

# --- Initial probability amplitude (real part) ---
psi_amplitude_real_initial = np.sqrt(gaussian_wavepacket_x_pdf(x_values, x0_initial, sigma_x))

# --- Animation Setup ---
fig, (ax_density, ax_phase) = plt.subplots(2, 1, figsize=(10, 10))

# Probability Density Plot
ax_density.set_xlim(x_values.min(), x_values.max())
ax_density.set_ylim(0, np.max(psi_amplitude_real_initial**2) * 1.1) # Fixed y-limit
ax_density.set_xlabel('Position $x$')
ax_density.set_ylabel(r'Probability Density $|ψ(x)|^2$')
ax_density.set_title(r'Effect of Boost Operator $\mathcal{B}(dp)$ on Wave Packet (Position Space)')
ax_density.grid(True)
line_density, = ax_density.plot([], [], 'r-', lw=2, label=r'$|ψ(x)|^2$')
p_net_text_density = ax_density.text(0.05, 0.9, '', transform=ax_density.transAxes, color='darkred', fontsize=12)

# Phase Plot
ax_phase.set_xlim(x_values.min(), x_values.max())
ax_phase.set_ylim(-np.pi * 1.1, np.pi * 1.1) # Y-limit for phase in radians
ax_phase.set_xlabel('Position $x$')
ax_phase.set_ylabel('Phase (radians)')
ax_phase.set_title('Quantum Phase of the Wave Packet')
ax_phase.grid(True)
ax_phase.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
ax_phase.set_yticklabels([r'$-\pi$', r'$-\pi/2$', r'$0$', r'$\pi/2$', r'$\pi$'])
line_phase, = ax_phase.plot([], [], 'b-', lw=2, label='Phase $\arg(\u03C8(x))$')
p_net_text_phase = ax_phase.text(0.05, 0.9, '', transform=ax_phase.transAxes, color='darkblue', fontsize=12)

plt.tight_layout()

# --- Animation Functions ---
def init_boost_phase_anim():
    line_density.set_data([], [])
    line_phase.set_data([], [])
    p_net_text_density.set_text('')
    p_net_text_phase.set_text('')
    return line_density, p_net_text_density, line_phase, p_net_text_phase

def update_boost_phase_anim(frame):
    current_p_net = p_net_start_anim + (p_net_end_anim - p_net_start_anim) * (frame / (num_frames_anim - 1))

    # Construct the complex wave packet amplitude for the current p_net
    # The magnitude part remains constant (gaussian_wavepacket_x_pdf with x0_initial, sigma_x)
    # The phase part is exp(i * current_p_net * x / hbar)
    psi_amplitude_real = np.sqrt(gaussian_wavepacket_x_pdf(x_values, x0_initial, sigma_x))
    psi_complex_current = psi_amplitude_real * np.exp(1j * current_p_net * x_values / hbar_val)

    # Probability density
    probability_density_current = np.abs(psi_complex_current)**2

    # Phase (using np.angle for sawtooth effect)
    phase_current = np.angle(psi_complex_current)

    # Update plots
    line_density.set_data(x_values, probability_density_current)
    line_phase.set_data(x_values, phase_current)

    # Update text
    p_net_text_density.set_text(fr'$<p_x> = {current_p_net:.2e}$ kg m/s')
    p_net_text_phase.set_text(fr'$<p_x> = {current_p_net:.2e}$ kg m/s')

    return line_density, p_net_text_density, line_phase, p_net_text_phase

# Create the animation
ani_boost_phase = FuncAnimation(fig, update_boost_phase_anim, frames=num_frames_anim,
                                init_func=init_boost_phase_anim, blit=True, interval=50)

# Display the animation
plt.close(fig) # Prevent static plot from showing
html_ani_boost_phase = HTML(ani_boost_phase.to_jshtml())
display(html_ani_boost_phase)

print("Saving boost phase animation as GIF...")
ani_boost_phase.save('boost_phase_wavepacket_animation.gif', writer='pillow', fps=20)
print("GIF saved: boost_phase_wavepacket_animation.gif")

The plots above illustrate a fundamental concept in quantum mechanics, directly related to the Heisenberg Uncertainty Principle:

*   **Position Space (Left Plot):** We started with a **narrow Gaussian wave packet** in position space (a small `sigma_x`). This represents a particle whose position is relatively well-defined or localized. As you can see, the probability density is sharply peaked around `x = 0`.

*   **Momentum Space (Right Plot):** The right plot shows the Fourier Transform of the position space wave packet, which represents the particle's probability distribution in momentum space. Notice that the momentum distribution is **much broader** than the position distribution. This indicates that while the particle's position is well-known, its momentum is consequently *less well-defined* or delocalized.

**Why the Difference?**

This inverse relationship between the spread in position ($\Delta x$) and the spread in momentum ($\Delta p$) is the essence of the Heisenberg Uncertainty Principle, which states $\Delta x \Delta p \ge \hbar/2$. A wave packet that is "clear" (narrow) in one representation (e.g., position) must necessarily be "unclear" (broad) in the other representation (momentum), and vice-versa. You cannot precisely know both the position and momentum of a particle simultaneously. Our visualization clearly shows that a localized state in position becomes a delocalized state in momentum.

Problem 29 animation colision

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftshift, fftfreq
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Set the animation embedding limit to a larger value (e.g., 50 MB) to avoid frame dropping
plt.rcParams['animation.embed_limit'] = 50.0

# --- Constants ---
hbar_val = 1.054571817e-34 # Reduced Planck constant (J*s)

# --- Define a 1D Gaussian wave packet in momentum space ---
def gaussian_wavepacket_p(p, p0, sigma_p):
    # Returns the probability amplitude (not density) for FT
    return (1 / (sigma_p * np.sqrt(2 * np.pi)))**0.5 * np.exp(-(p - p0)**2 / (4 * sigma_p**2))

# --- Simulation parameters ---
# Position space grid for visualization
x_max = 50
n_points_x = 2000 # Increased points for better resolution
x_values = np.linspace(-x_max, x_max, n_points_x)
dx = x_values[1] - x_values[0]

# Momentum space grid (determined by x_values via FFT)
p_max_ft = np.pi * hbar_val / dx # Max momentum from FFT
n_points_p = n_points_x
p_values_ft = fftshift(fftfreq(n_points_p, d=dx)) * (2 * np.pi * hbar_val)

# Parameters for the two 'jets' (momentum-space Gaussians)
sigma_p = 0.5 * hbar_val # Width of each jet in momentum space
p0_jet_initial_separation = 10 * hbar_val # Initial separation for p0

# --- Animation parameters ---
num_frames = 200
animation_duration = 3 # seconds (Decreased for faster animation)
interval_ms = 20 # milliseconds (Set to a fixed value for smoother, faster playback)

p0_jet_speed = p0_jet_initial_separation / (animation_duration * 0.9) # Adjusted speed to ensure collision within duration

# --- Animation Setup ---
fig, (ax_p, ax_x) = plt.subplots(1, 2, figsize=(16, 6))

# Momentum Space Plot (Left)
ax_p.set_xlim(p_values_ft.min() * 0.5, p_values_ft.max() * 0.5)
ax_p.set_ylim(0, (1 / (sigma_p * np.sqrt(2 * np.pi)))**0.5 * 1.1) # Max amplitude for p-space Gaussian
ax_p.set_xlabel('Momentum $p$')
ax_p.set_ylabel(r'Probability Amplitude $|\phi(p)|$')
ax_p.set_title('Momentum Space (Colliding Jets)')
ax_p.grid(True)
line_jet1, = ax_p.plot([], [], 'blue', label=r'Jet 1 $|\phi_1(p)|$')
line_jet2, = ax_p.plot([], [], 'green', label=r'Jet 2 $|\phi_2(p)|$')
line_total_p, = ax_p.plot([], [], 'red', linestyle='--', label=r'Total $|\phi_{total}(p)|$')
ax_p.legend()

# Position Space Plot (Right)
ax_x.set_xlim(-x_max / 2, x_max / 2) # Focus on central region
ax_x.set_ylim(0, 0.2) # Adjusted based on typical Gaussian amplitudes in x-space
ax_x.set_xlabel('Position $x$')
ax_x.set_ylabel(r'Probability Density $|\psi(x)|^2$')
ax_x.set_title('Position Space (Interference Pattern)')
ax_x.grid(True)
line_x, = ax_x.plot([], [], 'purple', label=r'$|\psi_{total}(x)|^2$')
ax_x.legend()

fig.suptitle('Conceptual Fourier Transform Animation: Colliding Momentum Jets', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle overlap

# --- Animation Functions ---
def init():
    line_jet1.set_data([], [])
    line_jet2.set_data([], [])
    line_total_p.set_data([], [])
    line_x.set_data([], [])
    return line_jet1, line_jet2, line_total_p, line_x

def update(frame):
    # Time-dependent p0 for the jets (moving towards each other)
    time = frame * (animation_duration / num_frames)
    current_separation = p0_jet_initial_separation - p0_jet_speed * time
    if current_separation < 0: current_separation = 0 # Ensure they don't pass each other too much

    p0_jet1 = current_separation / 2
    p0_jet2 = -current_separation / 2

    # Generate momentum space wave packets
    phi1_p = gaussian_wavepacket_p(p_values_ft, p0_jet1, sigma_p)
    phi2_p = gaussian_wavepacket_p(p_values_ft, p0_jet2, sigma_p)

    # Superposition in momentum space (complex amplitudes)
    phi_total_p = phi1_p + phi2_p

    # Update momentum space plot
    line_jet1.set_data(p_values_ft, np.abs(phi1_p))
    line_jet2.set_data(p_values_ft, np.abs(phi2_p))
    line_total_p.set_data(p_values_ft, np.abs(phi_total_p))

    # Inverse Fourier Transform to get position space wave function
    # Ensure normalization for ifft, the factor dx helps manage this
    # The ifft expects frequencies in 'standard' order (shifted=False), so we need to fftshift phi_total_p first
    # Also, scale by N to match numpy conventions for inverse transform magnitude
    psi_total_x_unshifted = ifft(fftshift(phi_total_p))
    psi_total_x = fftshift(psi_total_x_unshifted) * len(x_values) * dx # Normalization scaling

    # Probability density in position space
    probability_density_x = np.abs(psi_total_x)**2

    # Normalization check (optional, but good for debugging)
    # prob_norm_x = np.trapz(probability_density_x, x_values)
    # if not np.isclose(prob_norm_x, 1.0): print(f"Warning: Position space norm = {prob_norm_x}")

    # Update position space plot
    line_x.set_data(x_values, probability_density_x)
    ax_x.set_ylim(0, np.max(probability_density_x) * 1.1 if np.max(probability_density_x) > 0 else 0.2)

    return line_jet1, line_jet2, line_total_p, line_x

# Create the animation
ani_fourier_collision = FuncAnimation(fig, update, frames=num_frames,
                                       init_func=init, blit=True, interval=interval_ms)

# Display the animation
plt.close(fig) # Prevent static plot from showing
html_ani_fourier_collision = HTML(ani_fourier_collision.to_jshtml())
display(html_ani_fourier_collision)

print("Saving conceptual Fourier collision animation as GIF...")
ani_fourier_collision.save('fourier_collision_animation.gif', writer='pillow', fps=20)
print("GIF saved: fourier_collision_animation.gif")

### Conceptual Fourier Transform Animation: Colliding 'Particle Jets' in Momentum Space

This conceptual animation illustrates the profound connection between position space and momentum space through the Fourier transform, in the context of Heisenberg's Uncertainty Principle.

**Scenario:**

1.  **Momentum Space (Left):** We start with two Gaussian wave packets in momentum space, representing two 'particle jets' with opposite but equal magnitude central momentum values. These packets move towards each other, symbolizing a 'collision' or an increasing superposition of their momentum distributions. As they approach, they superimpose to form a total wave packet in momentum space.

2.  **Position Space (Right):** Simultaneously, we calculate the inverse Fourier transform of the total wave packet in momentum space to obtain its representation in position space. Initially, when the momentum jets are separated, the position distribution is broad and single. However, as the momentum jets superimpose, the probability distribution in position space begins to show **interference patterns**. These patterns are the direct result of the superposition of the probability amplitudes of the two jets, leading to distinctive peaks and valleys in the probability density in position space.

**Implication:**

This visualization highlights how momentum information (in this case, two distinct momentum components) translates into spatial interference patterns. A complex distribution in momentum space (the superposition of two crossing Gaussians) leads to a position distribution with fine structure. It is a visual manifestation of how probability amplitudes in one space combine to shape the distribution in the other space through the Fourier transform.